# Bölüm 14: RNN ve Attention ile Natural Language Processing (NLP)

**Hands-On Machine Learning with Scikit-Learn and PyTorch — Türkçe Notlar**

---

Alan Turing 1950'de ünlü **Turing Testi**'ni hayal ettiğinde, bir makinenin insan zekasını karşılayıp karşılayamayacağını değerlendirmek için bir yol önerdi. Resim tanıma, satranç oynama, müzik bestesi gibi birçok şeyi test edebilirdi; ancak ilginç şekilde dilsel bir görevi seçti: insanlarla sohbet eden ve onları insan olduğuna inandıran bir **chatbot**. Bu test, dili ustalıkla kullanmanın *Homo sapiens*'in en büyük bilişsel yeteneği olduğunu vurguluyor.

Son zamanlara kadar en iyi **Natural Language Processing (NLP)** modelleri **Recurrent Neural Networks (RNN)** tabanlıydı. Ancak son yıllarda RNN'ler büyük ölçüde **transformers** ile değiştirilmiştir (Bölüm 15). Yine de RNN'lerin NLP'de nasıl kullanıldığını öğrenmek önemlidir — hem transformerları anlamak için hem de bu bölümdeki teknikler (tokenization, beam search, attention mechanisms vb.) Transformer mimarileriyle de kullanışlıdır. Ayrıca RNN'ler yakın zamanda **state space models (SSMs)** biçiminde geri döndü.

## Bu Bölümün 3 Ana Konusu:

1. **Character RNN (Char-RNN):** Shakespeare metni üretmek — ilk küçük **language model**
2. **Sentiment Analysis:** IMDb film yorumlarını pozitif/negatif sınıflandırmak — Hugging Face kütüphaneleriyle
3. **Neural Machine Translation (NMT):** İngilizce'den İspanyolca'ya çeviri — **encoder-decoder** mimarisi ve **attention mechanisms**

In [ ]:
import sys

# Python sürümünün 3.10 veya daha yüksek olduğunu doğrula.
# Sürüm yetersizse AssertionError fırlatılır.
assert sys.version_info >= (3, 10)

In [ ]:
# Colab ortamındaysa torchmetrics kütüphanesini sessizce yükle
if IS_COLAB:
    %pip install -q torchmetrics

In [ ]:
from packaging.version import Version
import torch

# PyTorch sürümünün 2.6.0 veya daha yüksek olduğunu doğrula
assert Version(torch.__version__) >= Version("2.6.0")

In [ ]:
# Kullanılabilir en iyi donanım hızlandırıcıyı seç
if torch.cuda.is_available():
    device = "cuda"           # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = "mps"            # Apple Silicon GPU (M1/M2/M3)
else:
    device = "cpu"            # Hızlandırıcı yok → CPU kullanılır

device  # Seçilen cihazı göster

Donanım hızlandırıcı bulunamazsa kullanıcıyı uyar ve nasıl etkinleştireceği hakkında yönlendirme yap.

In [ ]:
if device == "cpu":
    print("Neural nets can be very slow without a hardware accelerator.")
    if IS_COLAB:
        # Colab'da GPU etkinleştirme yolu
        print("Go to Runtime > Change runtime and select a GPU hardware "
              "accelerator.")
    if IS_KAGGLE:
        # Kaggle'da GPU etkinleştirme yolu
        print("Go to Settings > Accelerator and select GPU.")

In [1]:
import matplotlib.pyplot as plt

# Grafik font boyutlarını global olarak ayarla
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

**`evaluate_tm()` ve `train()` yardımcı fonksiyonları:**

Bu iki fonksiyon tüm bölüm boyunca modelleri eğitmek ve değerlendirmek için tekrar tekrar kullanılır.

- `evaluate_tm(model, data_loader, metric)`: Modeli **evaluation mode**'a alır, gradyan hesaplamadan (torch.no_grad) veriyi geçirir ve bir TorchMetrics metriği döndürür.
- `train(...)`: Epoch döngüsünü, loss hesaplamayı, backpropagation'ı, learning rate scheduling'i (ReduceLROnPlateau) ve eğitim geçmişini (history) yönetir.
  - **ReduceLROnPlateau**: Validation metriği iyileşmezse learning rate'i `factor` kadar küçültür.
  - `epoch_callback`: Her epoch başında çağrılacak isteğe bağlı fonksiyon (örneğin stateful RNN'de hidden state sıfırlama için kullanılır).


In [2]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    """Modeli değerlendirme moduna alır ve verilen metriği hesaplar."""
    model.eval()        # Dropout ve BatchNorm'u değerlendirme moduna al
    metric.reset()      # Metriği sıfırla (önceki epoch'tan kalan değerleri temizle)
    with torch.no_grad():  # Gradient hesaplamayı devre dışı bırak (bellek ve hız için)
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()  # Toplam metrik değerini döndür

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    """
    Genel amaçlı eğitim döngüsü.
    
    Args:
        model: Eğitilecek PyTorch modeli
        optimizer: Ağırlık güncellemesi için optimizer (örn. NAdam)
        loss_fn: Kayıp fonksiyonu (örn. CrossEntropyLoss)
        metric: TorchMetrics metriği (örn. Accuracy)
        train_loader: Eğitim veri yükleyicisi
        valid_loader: Doğrulama veri yükleyicisi
        n_epochs: Kaç epoch eğitileceği
        patience: LR düşürmeden önce beklenecek epoch sayısı
        factor: LR'nin çarpılacağı küçültme faktörü (örn. 0.5 → yarıya indir)
        epoch_callback: Her epoch başında çağrılacak fonksiyon(opsiyonel)
    """
    # ReduceLROnPlateau: validation metriği iyileşmezse LR'yi düşür
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()  # Eğitim moduna al (Dropout aktif)
        
        # Epoch başında callback varsa çağır (örn. hidden state sıfırlama)
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()          # Backpropagation: gradyanları hesapla
            optimizer.step()         # Ağırlıkları güncelle
            optimizer.zero_grad()    # Gradyanları sıfırla (sonraki adım için)
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            # İlerleme bilgisini satır üzerine yaz
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)   # Val metriğine göre LR'yi ayarla
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history

**`del_vars()` – GPU RAM Yönetimi:**

Büyük modeller ve tensorlar GPU belleğini (VRAM) hızla doldurur. Bir modeli kullandıktan sonra `del model` yazmak yeterli değildir çünkü:
1. Python'un garbage collector'ı (çöp toplayıcı) hemen devreye girmeyebilir.
2. Jupyter/Colab'ın `Out` dictionary'si her hücrenin çıktısını saklar — model çıktıysa `Out`'ta da bir referans kalır.

Bu fonksiyon:
- Verilen değişken isimlerini `globals()`'tan siler
- Python'un `gc.collect()` ile garbage collection'ı zorla çalıştırır
- CUDA kullanıyorsak `torch.cuda.empty_cache()` ile GPU cache'ini boşaltır


In [3]:
import gc

def del_vars(variable_names=[]):
    """
    Belirtilen değişkenleri global scope'tan siler ve belleği temizler.
    
    Args:
        variable_names: Silinecek değişken isimlerinin listesi
    """
    for name in variable_names:
        try:
            del globals()[name]  # Global namespace'ten değişkeni sil
        except KeyError:
            pass  # Zaten silinmiş veya var olmayan değişkeni sessizce atla
    gc.collect()  # Python garbage collector'ı hemen çalıştır
    if device == "cuda":
        torch.cuda.empty_cache()  # CUDA cache'ini boşalt → VRAM'ı geri al

---
# BÖLÜM 1: Char-RNN ile Shakespeare Metni Üretimi

**Character RNN (Char-RNN):** Her karakter verildiğinde bir sonraki karakteri tahmin etmek için eğitilen bir RNN modelidir. Andrej Karpathy'nin 2015 tarihli "The Unreasonable Effectiveness of Recurrent Neural Networks" blog yazısında gösterildi.

Eğitildikten sonra model bir **language model** gibi davranarak yeni metin üretebilir. Aşağıda model tarafından üretilen örnek bir Shakespeare pasajı:

```
PANDARUS:
Alas, I think he shall be come approached and the day
When little srain would be attain'd into being never fed,
And who is but a chain and subjects of his death,
I should not sleep.
```

Bir şaheser değil, ama model sadece bir sonraki karakteri tahmin etmeyi öğrenerek kelimeler, dilbilgisi ve noktalama işaretleri öğrenebildi. Bu bizim ilk **language model** örneğimizdir.

## 1.1 Eğitim Veri Setinin Oluşturulması

Andrej Karpathy'nin ünlü **char-rnn** projesinden Shakespeare metnini indiriyoruz. Bu metin yaklaşık 1.1 milyon karakterden oluşur ve pek çok Shakespeare oyununun tam metnini içerir.

In [4]:
from pathlib import Path
import urllib.request

def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        urllib.request.urlretrieve(url, path)
    return path.read_text()

shakespeare_text = download_shakespeare_text()

In [5]:
# Metnin ilk 80 karakterini göster – nasıl göründüğünü kontrol etmek için
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


**Vocabulary (Kelime Dağarcığı) oluşturma:**

Character-level model için vocabulary; metinde geçen tüm benzersiz karakterlerin sıralanmış kümesidir.  
`sorted(set(...))` → hem tekrarları kaldırır hem alfabetik sıraya koyar.


In [6]:
vocab = sorted(set(shakespeare_text.lower()))
"".join(vocab)

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

**Character Encoding / Decoding (Karakter Kodlama):**

Sinir ağları sayılarla çalışır. Her karaktere benzersiz bir tam sayı ID atarız:
- `char_to_id`: Karakter → ID (örn. 'a' → 13)
- `id_to_char`: ID → Karakter (örn. 13 → 'a')

Bu iki sözlük birbirinin tersidir.


In [7]:
# Karakter → ID eşlemesi (forward lookup)
char_to_id = {char: index for index, char in enumerate(vocab)}
# ID → Karakter eşlemesi (reverse lookup)
id_to_char = {index: char for index, char in enumerate(vocab)}

In [8]:
# 'a' harfinin ID'sini göster
char_to_id["a"]

13

In [9]:
# ID 13'e karşılık gelen karakteri göster
id_to_char[13]

'a'

**`encode_text()` ve `decode_text()` fonksiyonları:**

- `encode_text(text)` → Metni karakter ID'lerinden oluşan bir PyTorch tensor'a çevirir.
- `decode_text(char_ids)` → Tensor'daki ID'leri tekrar orijinal metne dönüştürür.


In [10]:
import torch

def encode_text(text):
    """
    Metni karakter ID'lerinden oluşan bir PyTorch LongTensor'a çevirir.
    .lower() ile büyük/küçük harf normalleştirmesi yapılır.
    
    Örnek: 'Hello' → tensor([22, 14, 21, 21, 24])
    """
    return torch.tensor([char_to_id[char] for char in text.lower()])

def decode_text(char_ids):
    """
    Karakter ID tensor'ını orijinal metne geri çevirir.
    .item() ile tensor skalarını Python int'e dönüştürür.
    
    Örnek: tensor([22, 14, 21, 21, 24]) → 'hello'
    """
    return "".join([id_to_char[char_id.item()] for char_id in char_ids])

In [11]:
# 'Hello, world!' metnini encode et
encoded = encode_text("Hello, world!")
encoded

tensor([20, 17, 24, 24, 27,  6,  1, 35, 27, 30, 24, 16,  2])

In [12]:
# Encode edilen tensoru geri decode et
decode_text(encoded)

'hello, world!'

**`CharDataset` – Sliding Window Dataset:**

RNN eğitimi için **sliding window** (kayan pencere) yaklaşımını kullanıyoruz.

- `window_length = 50` → 50 karakterlik girdi penceresi
- Her pencere için **hedef (target)**, pencereyi bir karakter sola kaydırılmış versiyonudur
- Örnek: `x = "to be or n"` → `y = "o be or no"`

Bu yapı RNN'e "bu sıralamayı gördüğünde bir sonraki karakter nedir?" sorusunu öğretir.


In [13]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    """
    Sliding window yaklaşımıyla karakter dizisi veri seti.
    
    Her örnek:
    - x: [idx, idx+window_length) aralığındaki karakterler (girdi)
    - y: [idx+1, idx+window_length+1) aralığındaki karakterler (hedef)
    
    Yani y, x'ten bir adım ileride; RNN bir sonraki karakteri tahmin etmeyi öğrenir.
    """
    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)  # Tüm metni encode et
        self.window_length = window_length

    def __len__(self):
        # Son window'un taşmaması için toplam uzunluktan window boyutunu çıkar
        return len(self.encoded_text) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        end = idx + self.window_length
        window = self.encoded_text[idx : end]         # Girdi penceresi (x)
        target = self.encoded_text[idx + 1 : end + 1] # Hedef penceresi (y = x kaydırılmış)
        return window, target

**Train / Validation / Test veri setleri oluşturma:**

Shakespeare metni üç parçaya bölünür:
- **Train set**: İlk 1.000.000 karakter → modeli eğitmek için
- **Validation set**: 1.000.000 – 1.060.000 → hyperparameter ayarı ve erken durdurma için
- **Test set**: Geri kalan → son performans değerlendirmesi için

`DataLoader` batch'leri otomatik olarak oluşturur; `shuffle=True` eğitim sırasında batch sırasını karıştırır.


In [14]:
# --- VERİ YÜKLEYİCİLER ---
# Metnin %90'ı eğitim (1 milyon karakter), %5 validation, %5 test

window_length = 50      # RNN 50 karakterlik bağlamı görecek
batch_size = 512        # GPU'ya sığmazsa azaltabilirsiniz

train_set   = CharDataset(shakespeare_text[:1_000_000], window_length)          # ~90%
valid_set   = CharDataset(shakespeare_text[1_000_000:1_060_000], window_length) # ~5%
test_set    = CharDataset(shakespeare_text[1_060_000:], window_length)           # ~5%

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)  # Shuffle: eğitimde karıştır
valid_loader = DataLoader(valid_set, batch_size=batch_size)                # Validation: sıralı
test_loader  = DataLoader(test_set, batch_size=batch_size)                 # Test: sıralı

# Her batch: [512 adet 50 karakterlik window, her pencere için 50 karakterlik target]
# Dikkat: window_length büyük → daha uzun bağlam öğrenir, ama eğitim daha zor
# window_length küçük → daha hızlı eğitim, ama uzun kalıpları öğrenemez

In [15]:
# CharDataset'i kısa bir örnekle dene
# 'To be or not to be' → window_length=10 ile sliding window gösterimi
to_be_dataset = CharDataset("To be or not to be", window_length=10)
for x, y in to_be_dataset:
    print(f"x={x}, y={y}")
    print(f"    decoded: x={decode_text(x)!r}, y={decode_text(y)!r}")

x=tensor([32, 27,  1, 14, 17,  1, 27, 30,  1, 26]), y=tensor([27,  1, 14, 17,  1, 27, 30,  1, 26, 27])
    decoded: x='to be or n', y='o be or no'
x=tensor([27,  1, 14, 17,  1, 27, 30,  1, 26, 27]), y=tensor([ 1, 14, 17,  1, 27, 30,  1, 26, 27, 32])
    decoded: x='o be or no', y=' be or not'
x=tensor([ 1, 14, 17,  1, 27, 30,  1, 26, 27, 32]), y=tensor([14, 17,  1, 27, 30,  1, 26, 27, 32,  1])
    decoded: x=' be or not', y='be or not '
x=tensor([14, 17,  1, 27, 30,  1, 26, 27, 32,  1]), y=tensor([17,  1, 27, 30,  1, 26, 27, 32,  1, 32])
    decoded: x='be or not ', y='e or not t'
x=tensor([17,  1, 27, 30,  1, 26, 27, 32,  1, 32]), y=tensor([ 1, 27, 30,  1, 26, 27, 32,  1, 32, 27])
    decoded: x='e or not t', y=' or not to'
x=tensor([ 1, 27, 30,  1, 26, 27, 32,  1, 32, 27]), y=tensor([27, 30,  1, 26, 27, 32,  1, 32, 27,  1])
    decoded: x=' or not to', y='or not to '
x=tensor([27, 30,  1, 26, 27, 32,  1, 32, 27,  1]), y=tensor([30,  1, 26, 27, 32,  1, 32, 27,  1, 14])
    decoded: x=


## 1.2 Embeddings — Kategorik Özelliklerin Yoğun Temsili

**Neden token ID'lerini doğrudan kullanamayız?**

Çoğu ML modeli, benzer girdilerin benzer şeyleri temsil ettiğini varsayar. Ancak ID'ler rastgele atanır — ID 5 ve ID 6 birbirine hiç benzemeyen karakterleri temsil edebilir. Model bu yüzeysel yakınlık tarafından yanıltılır.

**Çözüm 1: One-Hot Encoding**
Tüm vektörler birbirinden eşit uzaklıkta. Vocabulary 39 karakter → her karakter 39 boyutlu one-hot vektör. İdare edilir, ama kelimelerle çalışırken vocabulary on binlere çıkabilir → impractical.

**Çözüm 2: Embeddings (Daha İyi!)**
**Embedding**: kategorik bir özelliğin yoğun (dense) temsilidir. 50.000 kategoride one-hot → 50.000 boyutlu seyrek vektör. Embedding ise sadece 300 boyutlu küçük dense vektör.

- Başlangıçta rastgele başlatılır
- Gradient descent ile diğer model parametreleriyle birlikte eğitilir
- Benzer kategoriler birbirine yakın embedding'lere sahip olur → **representation learning**

**Ünlü Word Embedding Örneği (Mikolov, Google, 2013):**
- *King – Man + Woman ≈ Queen* → embedding'ler cinsiyet kavramını öğrendi!
- *Madrid – Spain + France ≈ Paris* → başkent kavramı kodlanmış!

**Önemli Not:** Word embedding'ler bazen önyargıları da kodlar. Örneğin *Man:Doctor :: Woman:Nurse* gibi cinsiyetçi kalıplar. Bu, derin öğrenmede adalet ve tarafsızlık araştırmasının önemli bir konusudur.

**Embedding boyutu:** İyi bir kural: `sqrt(n_categories)` civarında bir değer iyi çalışır.

In [16]:
import torch.nn as nn

# --- nn.Embedding KULLANIMI ---
# nn.Embedding(num_categories, embedding_dim) → embedding matrix: [num_categories × embedding_dim]
# Lookup tablosu gibi çalışır: kategori ID → ilgili satırı döndürür
# One-hot encoding + Linear layer işlemine matematiksel olarak eşdeğer,
# ama çok daha verimli (sıfırlarla gereksiz çarpma yok)

torch.manual_seed(42)
embed = nn.Embedding(5, 3)  # 5 kategori, 3 boyutlu embeddings
embed(torch.tensor([[3, 2], [0, 2]]))


# Dikkat: Kategori 2 iki kez göründü ve her seferinde aynı embedding döndü
# Embedding'ler henüz eğitilmedi → rastgele değerler

tensor([[[ 0.2674,  0.5349,  0.8094],
         [ 2.2082, -0.6380,  0.4617]],

        [[ 0.3367,  0.1288,  0.2345],
         [ 2.2082, -0.6380,  0.4617]]], grad_fn=<EmbeddingBackward0>)

---
## 1.3 Char-RNN Modelini Oluşturma ve Eğitme



**ShakespeareModel mimarisi:**

```
Girdi (karakter ID'leri)
    ↓
Embedding katmanı  [vocab_size → embed_dim=10]
    ↓
GRU (2 katmanlı)   [embed_dim → hidden_dim=128]
    ↓
Linear (Output)    [hidden_dim → vocab_size]
    ↓
Çıktı logits: her pozisyon için vocab_size'lık olasılık dağılımı
```

**GRU (Gated Recurrent Unit):**
- LSTM'in daha basit ve hızlı bir varyantı
- **Reset gate** ve **Update gate** ile uzun vadeli bağımlılıkları öğrenir
- `batch_first=True` → girdi şekli (batch, sequence, features) olur
- `dropout=0.1` → regularization için her katman arasında %10 neuron devre dışı

**`forward()` açıklaması:**
1. `embed(X)` → Her karakter ID'sini embedding vektörüne dönüştür
2. `gru(embeddings)` → Tüm zaman adımlarındaki çıktıları hesapla
3. `output(outputs)` → Her zaman adımı için vocab_size logit üret
4. `.permute(0, 2, 1)` → CrossEntropyLoss için shape'i (B, vocab_size, seq_len) yap


In [17]:
# --- ShakespeareModel: Char-RNN ---
# Mimari:
# 1. nn.Embedding → karakter ID'lerini dense vektörlere dönüştürür
# 2. nn.GRU (2 katman, 128 hidden unit) → sequential context öğrenir
# 3. nn.Linear → her time step'te 39 karakterden birini tahmin eder

class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128, dropout=0.1):
        super().__init__()
        # Embedding katmanı: vocab_size karakter → embed_dim boyutlu dense vektörler
        self.embed = nn.Embedding(vocab_size, embed_dim)
        # GRU: embed_dim giriş, hidden_dim hidden, n_layers katman
        # batch_first=True: [batch, seq, features] formatı
        # dropout: aşırı öğrenmeyi önler (yalnızca katmanlar arası)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        # Output: her time step'te vocab_size logit üret
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X):
        embeddings = self.embed(X)          # [batch, window, embed_dim]
        outputs, _states = self.gru(embeddings)  # [batch, window, hidden_dim]
        # nn.CrossEntropyLoss class dimension'ın 2. boyutta olmasını bekler
        # → permute(0, 2, 1): [batch, hidden_dim, window] → [batch, vocab_size, window]
        return self.output(outputs).permute(0, 2, 1)

torch.manual_seed(42)
vocab_size = len(vocab)  # 39
model = ShakespeareModel(vocab_size)

# Eğitim: nn.CrossEntropyLoss + Adam optimizer
# Accuracy (karakter düzeyinde doğruluk) metriği ile değerlendirilir

**⚠️ Uyarı:** Aşağıdaki eğitim hücresi **GPU olmadan çok yavaş çalışır** (20 epoch, ~1 milyon karakter).  
GPU kullanıyorsanız birkaç dakika, CPU ile saatler sürebilir.

**Optimizer ve Loss:**
- **NAdam**: Adam optimizer'ının Nesterov momentum eklenmiş versiyonu — genellikle daha hızlı yakınsama
- **CrossEntropyLoss**: Çok sınıflı sınıflandırma için standart kayıp fonksiyonu

**Accuracy metriği:**
- `task="multiclass"` → Her pozisyonda vocab_size arasından doğru karakteri seçme


In [18]:
# Donanımı belirle (GPU varsa kullan, yoksa CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_epochs = 20
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters())

# Accuracy nesnesini tanımlarken artık 'device' hata vermeyecek
accuracy = torchmetrics.Accuracy(task="multiclass", 
                                 num_classes=len(vocab)).to(device)

# Modeli eğit
history = train(model, optimizer, xentropy, accuracy, train_loader, valid_loader, 
                n_epochs)

Epoch 1/20,                      train loss: 1.6041, train metric: 51.28%, valid metric: 52.16%
Epoch 2/20,                      train loss: 1.3855, train metric: 56.67%, valid metric: 52.66%
Epoch 3/20,                      train loss: 1.3556, train metric: 57.41%, valid metric: 53.84%
Epoch 4/20,                      train loss: 1.3411, train metric: 57.78%, valid metric: 53.98%
Epoch 5/20,                      train loss: 1.3326, train metric: 58.00%, valid metric: 54.10%
Epoch 6/20,                      train loss: 1.3269, train metric: 58.14%, valid metric: 53.85%
Epoch 7/20,                      train loss: 1.3230, train metric: 58.24%, valid metric: 54.50%
Epoch 8/20,                      train loss: 1.3200, train metric: 58.32%, valid metric: 54.41%
Epoch 9/20,                      train loss: 1.3176, train metric: 58.38%, valid metric: 54.39%
Epoch 10/20,                      train loss: 1.3155, train metric: 58.42%, valid metric: 54.41%
Epoch 11/20,                      train

In [21]:
# Eğitilmiş modelin ağırlıklarını diske kaydet
# state_dict() → modelin tüm öğrenilebilir parametrelerini içeren dictionary
torch.save(model.state_dict(), "my_shakespeare_model.pt")

**Modeli test etme:** Eğitilen modele "To be or not to b" girişini verip bir sonraki karakteri tahmin edelim. Beklenen cevap: **'e'**

**`unsqueeze(dim=0)`:** Modelin beklendiği (batch, seq) formatı için batch boyutu ekler.  
**`argmax()`:** Logits içindeki en yüksek değerin indeksini verir → en olası karakter.


In [22]:
model.eval()  # Değerlendirme moduna al (Dropout devre dışı)
text = "To be or not to b"
# encode_text: karakter → ID | unsqueeze: (seq,) → (1, seq) batch boyutu ekle
encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
with torch.no_grad():  # Gradient hesaplamadan forward pass
    Y_logits = model(encoded_text)
    # Son pozisyondaki en yüksek logit → tahmin edilen karakter ID'si
    predicted_char_id = Y_logits[0, :, -1].argmax().item()
    predicted_char = id_to_char[predicted_char_id]  # Doğru tahmin: "e"


In [23]:
# Tahmin edilen karakteri göster (beklenen: 'e')
predicted_char

'e'

## 1.4 Shakespearean Metin Üretme

**Stochastic Sampling (Stokastik Örnekleme):**

Greedy decoding (her adımda en olası karakteri seçmek) çok tekrarcı metinler üretir. Bunun yerine `torch.multinomial` ile olasılık dağılımından örnekleme yaparız.

**Temperature (Sıcaklık) parametresi:**
- `temperature < 1` → Dağılımı keskinleştirir (daha deterministik, kısmen sıkıcı)
- `temperature = 1` → Orijinal model dağılımı
- `temperature > 1` → Dağılımı düzleştirir (daha çeşitli ama daha anlamsız)

**Formül:** `softmax(logits / temperature)`  
Düşük temperature → yüksek logitler çok daha olası olur.


In [24]:
torch.manual_seed(42)
# Örnek olasılık dağılımı: 3 kategori, %50 / %40 / %10 olasılıkla
probs = torch.tensor([[0.5, 0.4, 0.1]])  # probas = 50%, 40%, and 10%
# multinomial: bu dağılımdan yerine koyarak (replacement) 8 örnek al
samples = torch.multinomial(probs, replacement=True, num_samples=8)
samples

tensor([[0, 0, 0, 0, 1, 0, 2, 2]])

In [25]:
import torch.nn.functional as F

def next_char(model, text, temperature=1):
    """
    Verilen metinden bir sonraki karakteri stokastik olarak örnekler.
    
    Args:
        model: Eğitilmiş ShakespeareModel
        text: Bağlam metni (son karakterler)
        temperature: Örnekleme sıcaklığı
                     < 1 → daha deterministik (tekrarcı ama tutarlı)
                     = 1 → orijinal model dağılımı
                     > 1 → daha rastgele (çeşitli ama anlamsız)
    
    Returns:
        str: Tahmin edilen tek karakter
    """
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        # Son pozisyonun logitlerini temperature ile böl → dağılımı ayarla
        Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
        # Olasılık dağılımından bir örnek al
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()
    return id_to_char[predicted_char_id]

In [26]:
def extend_text(model, text, n_chars=80, temperature=1):
    """
    Verilen başlangıç metnini n_chars karakter kadar uzatır.
    Her adımda bir sonraki karakteri tahmin ederek metne ekler.
    
    Args:
        model: Eğitilmiş karakter modeli
        text: Başlangıç metni (seed text)
        n_chars: Üretilecek karakter sayısı
        temperature: Örnekleme sıcaklığı
    
    Returns:
        str: Genişletilmiş metin
    """
    for _ in range(n_chars):
        text += next_char(model, text, temperature)
    return text

In [27]:
# Çok düşük temperature (0.01) → neredeyse deterministik, tekrarcı
# Model her seferinde en olası karakteri seçer
print(extend_text(model, "To be or not to b", temperature=0.01))

To be or not to be a stranger with the stroke
of the wars to the common person and the strong sta


In [28]:
# Orta temperature (0.4) → dengeli ve okunabilir Shakespeare üslubu
print(extend_text(model, "To be or not to b", temperature=0.4))

To be or not to be with me to be
my father is he that is a fool,
and then i will not see him bear


In [29]:
# Çok yüksek temperature (100) → tamamen rastgele, anlamsız metin
# Tüm karakterler neredeyse eşit olasılıkla seçilir
print(extend_text(model, "To be or not to b", temperature=100))

To be or not to b ;cv.
smivx?b3
nz3d k 'pulvcly'pm-mwlgew&$b3&'qkrau3x 't!hsfd $?i?mu3l!lgrdgjx$n


## 1.5 Stateful RNN Eğitimi

### Stateless vs Stateful RNN

**Stateless RNN (Durumsuz RNN):**
- Her eğitim iteration'ında hidden state sıfırdan (zeros) başlar
- Son adımın hidden state'i atılır
- Yalnızca `window_length` uzunluğundaki bağlam öğrenilebilir

**Stateful RNN (Durumlu RNN):**
- Bir batch sonunda hidden state korunur
- Sonraki batch bu son hidden state'ten başlar
- Böylece model `window_length`'ten çok daha uzun döngüsel örüntüler öğrenebilir

**Stateful RNN'in zorluğu:** Dataset hazırlamak kritiktir!  
n. batch'teki n. sekans, (n-1). batch'teki n. sekansın hemen devamı olmalıdır.


Şimdiye kadar yalnızca stateless RNNs kullandık: her eğitim iterasyonunda model, tamamen sıfırlardan oluşan bir hidden state ile başlar, ardından her time step’te bu durumu günceller ve son time step’ten sonra bu durumu atar, çünkü artık gerekli değildir.

Peki, RNN’e eğitim batch’i işlendikten sonra bu final state’i saklamasını ve bir sonraki batch için initial state olarak kullanmasını söyleseydik ne olurdu?

Böylece model, yalnızca kısa diziler üzerinden backpropagation yapsa bile, uzun vadeli örüntüleri (long-term patterns) öğrenebilirdi. Bu yaklaşıma stateful RNN denir. Şimdi nasıl inşa edileceğine bakalım.

⸻

Modelin kendisinde çok az değişiklik gerekir:

1️⃣ Yeni bir hidden_states attribute ekleyeceğiz ve bunu None olarak initialize edeceğiz.

2️⃣ Her batch işlendikten sonra hidden states’i kaydedeceğiz ve bir sonraki batch için initial hidden states olarak kullanacağız.

⚠️ Dikkat: Bu hidden states üzerinde detach() çağrısı yapmalıyız. Böylece bir sonraki iterasyonda bu eğitim iterasyonunun computation graph’ı üzerinden backpropagation yapılmaz (aksi takdirde hata oluşur).

In [30]:
class StatefulShakespeareModel(nn.Module):
    """
    Durumlu (Stateful) Shakespearean metin modeli.
    
    Stateless modelden tek farkı:
    - `self.hidden_states` attribute'u ile hidden state'i batchler arası saklar
    - forward() sonunda hidden state'i detach() ile hesaplama grafiğinden ayırır
    
    ÖNEMLİ: detach() olmadan bir sonraki batch'te backprop hata verir
    (eski hesaplama grafiği üzerinden gradient hesaplamaya çalışır).
    """
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128,
                 dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)
        self.hidden_states = None  # Başlangıçta sıfır (GRU otomatik sıfırlar)

    def forward(self, X):
        embeddings = self.embed(X)
        # Önceki batch'ten kalan hidden state'i kullan (ilk batch'te None → GRU sıfırlar)
        outputs, hidden_states = self.gru(embeddings, self.hidden_states)
        # Hidden state'i koru ama hesaplama grafiğinden ayır (bellek sızıntısını önle)
        self.hidden_states = hidden_states.detach()
        return self.output(outputs).permute(0, 2, 1)

Stateful RNN’lerdeki en büyük zorluk, dataset hazırlamaktır.

Gerçekte, bir stateful RNN yalnızca mantıklıdır, eğer batch içindeki her input sequence, önceki batch’teki ilgili sequence’in tam olarak bıraktığı yerden başlıyorsa.

Daha açık söylemek gerekirse: batch k içindeki nth window, batch k – 1 içindeki nth window’un durduğu yerden başlamalıdır.

Örnek: Diyelim ki tam encoded text şöyle:

Window length = 4 ve batch size = 5 olsun. Dataset şu şekilde 3 batch içerebilir:

In [ ]:
Batch #1:
X = [[1,2,3,4], [13,14,15,16], [25,26,27,28], [37,38,39,40], [49,50,51,52]]
Y = [[2,3,4,5], [14,15,16,17], [26,27,28,29], [38,39,40,41], [50,51,52,53]]

Batch #2:
X = [[5,6,7,8], [17,18,19,20], [29,30,31,32], [41,42,43,44], [53,54,55,56]]
Y = [[6,7,8,9], [18,19,20,21], [30,31,32,33], [42,43,44,45], [54,55,56,57]]

Batch #3:
X = [[9,10,11,12], [21,22,23,24], [33,34,35,36], [45,46,47,48], [57,58,59,60]]
Y = [[10,11,12,13], [22,23,24,25], [34,35,36,37], [46,47,48,49], [58,59,60,61]]

Şimdi, verileri bu şekilde organize eden bir StatefulCharDataset class’ı yazalım.

In [32]:
from torch.utils.data import Dataset, DataLoader

class StatefulCharDataset(Dataset):
    """
    Stateful RNN için tasarlanmış veri seti.
    
    Anahtar kural: k. batch'teki n. sekans, (k-1). batch'teki n. sekansın
    hemen devamı olmalıdır. Bunu sağlamak için metin sabit aralıklı 
    (spacing) başlangıç noktalarına bölünür.
    
    Örnek (window_length=4, batch_size=5, text=[1..60]):
    Batch #1: X=[[1,2,3,4], [13,14,15,16], [25,..], [37,..], [49,..]]
    Batch #2: X=[[5,6,7,8], [17,18,19,20], [29,..], [41,..], [53,..]]
    
    Her batch'in n. elemanı bir öncekinin n. elemanının devamıdır.
    """
    def __init__(self, text, window_length, batch_size):
        self.encoded_text = encode_text(text)
        self.window_length = window_length
        self.batch_size = batch_size
        # Toplam ardışık pencere sayısı
        n_consecutive_windows = (len(self.encoded_text) - 1) // window_length
        # Her 'slot' (batch pozisyonu) için pencere sayısı
        n_windows_per_slot = n_consecutive_windows // batch_size
        self.length = n_windows_per_slot * batch_size
        # Slot'lar arasındaki karakter aralığı
        self.spacing = n_windows_per_slot * window_length

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        # idx % batch_size → hangi slot (sütun), idx // batch_size → kaçıncı pencere
        start = ((idx % self.batch_size) * self.spacing
                 + (idx // self.batch_size) * self.window_length)
        end = start + self.window_length
        window = self.encoded_text[start : end]
        target = self.encoded_text[start + 1 : end + 1]
        return window, target

**Stateful DataLoader ayarları:**

Stateful RNN için DataLoader'da iki önemli kural vardır:
1. **`shuffle=False`**: Batch sırası karıştırılmamalı (batchler arası süreklilik kırılır)
2. **`drop_last=True`**: Tüm batchlerin tam olarak aynı boyutta olmasını garantiler (son eksik batch atılır)


In [33]:
batch_size = 128
# Eğitim seti (ilk 1 milyon karakter)
stateful_train_set = StatefulCharDataset(shakespeare_text[:1_000_000],
                                         window_length, batch_size)
# shuffle=False → sıralama bozulmamalı, drop_last=True → eksik son batch atılsın
stateful_train_loader = DataLoader(stateful_train_set, batch_size=batch_size,
                                   drop_last=True)

# Doğrulama seti
stateful_valid_set = StatefulCharDataset(shakespeare_text[1_000_000:1_060_000],
                                         window_length, batch_size)
stateful_valid_loader = DataLoader(stateful_valid_set, batch_size=batch_size,
                                   drop_last=True)

# Test seti
stateful_test_set = StatefulCharDataset(shakespeare_text[1_060_000:],
                                        window_length, batch_size)
stateful_test_loader = DataLoader(stateful_test_set, batch_size=batch_size,
                                  drop_last=True)

**`epoch_callback` ile hidden state sıfırlama:**

Her epoch başında hidden state'i sıfırlamak gerekir. Bunun için `train()` fonksiyonunun `epoch_callback` parametresine bir fonksiyon geçilir. Bu yaklaşım eğitim döngüsünü değiştirmeden esneklik sağlar.


**⚠️ Uyarı:** Aşağıdaki hücre özellikle GPU olmadan çok uzun sürebilir.

In [34]:
torch.manual_seed(42)

stateful_model = StatefulShakespeareModel(len(vocab)).to(device)

n_epochs = 10
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(stateful_model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=len(vocab)).to(device)

def reset_hidden_states(model, epoch):
    """Her epoch başında çağrılır; hidden state sıfırlanır."""
    model.hidden_states = None  # None verilince GRU otomatik sıfırlar

# epoch_callback ile her epoch başında hidden state sıfırlanacak
history = train(stateful_model, optimizer, xentropy, accuracy, stateful_train_loader,
                stateful_valid_loader, n_epochs, epoch_callback=reset_hidden_states)

Epoch 1/10,                      train loss: 2.4729, train metric: 29.87%, valid metric: 39.21%
Epoch 2/10,                      train loss: 1.8774, train metric: 44.50%, valid metric: 44.88%
Epoch 3/10,                      train loss: 1.6903, train metric: 49.32%, valid metric: 47.58%
Epoch 4/10,                      train loss: 1.5954, train metric: 51.78%, valid metric: 50.08%
Epoch 5/10,                      train loss: 1.5384, train metric: 53.23%, valid metric: 51.20%
Epoch 6/10,                      train loss: 1.5009, train metric: 54.19%, valid metric: 52.15%
Epoch 7/10,                      train loss: 1.4720, train metric: 54.90%, valid metric: 52.62%
Epoch 8/10,                      train loss: 1.4509, train metric: 55.44%, valid metric: 53.09%
Epoch 9/10,                      train loss: 1.4333, train metric: 55.88%, valid metric: 53.58%
Epoch 10/10,                      train loss: 1.4197, train metric: 56.25%, valid metric: 53.73%


In [35]:
# Stateful modeli kaydet
torch.save(stateful_model.state_dict(), "my_stateful_shakespeare_model.pt")

**Stateful RNN ile metin üretme:**

Stateful RNN kullanırken metin üretimi de farklıdır:
- Hidden state başta bir kez sıfırlanır
- Her adımda yalnızca **son karakter** model girdisi olarak verilir (tam metin değil)
- Model önceki hidden state'i korur ve onun üzerine yeni karakteri işler


In [36]:
def extend_text_with_stateful_rnn(model, text, n_chars=80, temperature=1):
    """
    Stateful RNN kullanarak metin üretir.
    
    Stateless versiyondan farkı:
    - Hidden state başta sıfırlanır, sonra korunur
    - Her adımda yalnızca son karakter girilir (tam bağlam değil)
    - Model geçmişi hidden state üzerinden taşır
    """
    model.hidden_states = None  # Başlangıçta hidden state sıfırla
    rnn_input = text             # İlk girdi tüm bağlam metni
    for _ in range(n_chars):
        char = next_char(model, rnn_input, temperature)
        text += char
        rnn_input = char  # Sonraki adımda yalnızca son karakteri ver
    return text + "…"    # Metnin devam ettiğini belirtmek için "…" ekle

In [37]:
torch.manual_seed(42)
stateful_model.eval()
# Düşük temperature: tutarlı ama tekrarcı
print(extend_text_with_stateful_rnn(stateful_model, "To be or not to b",
                                    temperature=0.1))

To be or not to be the content
that the senate the country to the consent.

clarence:
what is the…


In [38]:
# Orta temperature: dengeli üretim
print(extend_text_with_stateful_rnn(stateful_model, "To be or not to b", temperature=0.4))

To be or not to bed
and my lord, and the bearding the duke is heaven
that i shall not be thine en…


In [39]:
# Yüksek temperature: çeşitli ama daha anlamsız
print(extend_text_with_stateful_rnn(stateful_model, "To be or not to b", temperature=1))

To be or not to beal is the sun.

king richard ii:
i knops dily, marry, which 'to, how conscider?…


GPU RAM'ini temizle:

In [40]:
Out.clear()  # Jupyter'ın Out dictionary'sini temizle (büyük tensor referanslarını sil)
del_vars(["accuracy", "embed", "encoded", "encoded_text", "optimizer", "probs",
          "samples", "x", "y", "shakespeare_text", "stateful_test_loader",
          "stateful_train_loader", "Y_logits", "stateful_valid_loader",
          "test_loader", "train_loader", "valid_loader", "xentropy"])

# 2. Sentiment Analysis (Duygu Analizi)

**Sentiment Analysis nedir?**  
Bir metindeki duygusal tonu (pozitif / negatif / nötr) otomatik olarak sınıflandırma görevidir.

**Bu bölümde kullanılacak veri seti: IMDB**
- 50.000 film yorumu
- İkili sınıflandırma: **pozitif (1)** veya **negatif (0)**
- Gerçek dünya NLP görevi için ideal bir benchmark

**Bu bölümde işlenecek konular:**
1. IMDB veri setini yükleme
2. Tokenization yöntemleri (BPE, BBPE, WordPiece, Unigram LM)
3. Sıfırdan RNN tabanlı sentiment model oluşturma
4. Pack/Pad mekanizması ile değişken uzunluklu sekanslar
5. Bidirectional RNN
6. Pretrained BERT embeddings ve fine-tuning
7. Transformers Trainer API
8. Inference Pipeline'ları


## 2.1 IMDB Veri Setini Yükleme

Hugging Face `datasets` kütüphanesini kullanarak IMDB film yorumları veri setini yüklüyoruz.

In [ ]:
%pip install "aiosignal<1.4.0"

  Using cached aiosignal-1.3.2-py2.py3-none-any.whl.metadata (3.8 kB)


In [1]:
from datasets import load_dataset

# Hugging Face Hub'dan IMDB veri setini yükle
imdb_dataset = load_dataset("imdb")

# Eğitim setini %80 train / %20 validation olarak böl (seed=42 tekrarlanabilirlik için)
split = imdb_dataset["train"].train_test_split(train_size=0.8, seed=42)
imdb_train_set, imdb_valid_set = split["train"], split["test"]
imdb_test_set = imdb_dataset["test"]

ImportError: cannot import name 'load_dataset' from 'datasets' (unknown location)

In [ ]:
# Eğitim setindeki 2. yorumu göster (indeks 1)
imdb_train_set[1]["text"]

In [ ]:
# 2. yorumun etiketini göster (0=negatif, 1=pozitif)
imdb_train_set[1]["label"]

In [ ]:
# 17. yorumu göster
imdb_train_set[16]["text"]

In [ ]:
# 17. yorumun etiketini göster
imdb_train_set[16]["label"]

## 2.2 Tokenization – `tokenizers` Kütüphanesi ile

**Tokenization nedir?**  
Ham metni modelin işleyebileceği sayısal birimlere (token'lara) dönüştürme işlemidir.

**Neden karakter yerine token?**
- Karakter düzeyi: Çok uzun sekanslar, bağlamı zor yakalar
- Kelime düzeyi: Bilinmeyen kelimeler (OOV), çok büyük vocabulary
- **Subword tokenization** (alt-kelime): İkisinin de avantajlarını alır

### Başlıca Subword Tokenization Yöntemleri:

| Yöntem | Kullandığı Modeller | Açıklama |
|--------|---------------------|----------|
| **BPE** (Byte-Pair Encoding) | GPT-1 | En sık geçen karakter çiftlerini birleştirir |
| **BBPE** (Byte-level BPE) | GPT-2, RoBERTa | BPE'yi byte düzeyinde uygular, Unicode sorunsuz |
| **WordPiece** | BERT | BPE benzeri ama birleştirme kriteri farklı |
| **Unigram LM** | ALBERT, T5, XLM-R | Olasılıksal, birden fazla tokenization seçeneği |


### Diğer Tokenization Algoritmaları

**Byte-Level BPE (BBPE):**
- `ByteLevel` pre-tokenizer boşlukları özel `Ġ` karakteriyle değiştirir → BPE boşlukların nerede olduğunu bilir
- Byte seviyesinde çalışır → 256 byte vocabulary'de → asla `<unk>` üretmez
- Emoji gibi nadir karakterler için ideal

**WordPiece (Google, 2016):**
- BPE'ye benzer, ama en sık çifti değil, en yüksek **skoru** olan çifti vocabulary'ye ekler
- Score = frequency(AB) / (freq(A) × freq(B)) × len(vocab)
- Payda: bireysel tokenlar zaten sık ise penalize eder → daha anlamlı tokenlar
- BPE'den daha kısa encode dizileri üretir
- Kelime içi tokenları `##` öneki ile işaretler → kolayca birleştirilebilir
- BERT tarafından kullanılır

**Unigram LM (Taku Kudo, Google, 2018):**
- Tersine çalışır: büyük vocabulary ile başla, en az kullanışlı tokenları kaldır
- Token seçiminde Unigram dil modeli varsayımı (tokenlar bağımsız örneklenir)
- Özellikle boşluk kullanmayan dillerde iyi çalışır (Çince, Japonca vb.)
- **Subword Regularization:** eğitim sırasında aynı kelimeyi farklı şekillerde tokenize et → daha iyi genelleme
  - Örn: "them" → bazen ["them"], bazen ["the", "m"]
  - Morfemik olarak zengin dillerde (Türkçe, Arapça, Almanca, vb.) çok etkili!

### 2.2.1 BPE (Byte-Pair Encoding) Tokenizer Eğitme

In [42]:
import tokenizers

# BPE modeli oluştur
# unk_token: bilinmeyen token'lar için özel sembol
bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)

# Pre-tokenizer: tokenizer çalışmadan önce boşluklara göre ön bölme yapar
# 'Hello, world!!!' → [('Hello,', (0,6)), ('world!!!', (7,15))]
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()

# Özel token'lar: <pad> padding için, <unk> bilinmeyen kelimeler için
special_tokens = ["<pad>", "<unk>"]

# BPE Trainer: vocabulary boyutu 1000 olacak şekilde birleştirme kuralları öğrenir
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000,
                                             special_tokens=special_tokens)

# IMDB yorumlarını lowercase yaparak eğitim için hazırla
train_reviews = [review["text"].lower() for review in imdb_train_set]

# BPE tokenizer'ı IMDB verisi üzerinde eğit
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

ModuleNotFoundError: No module named 'tokenizers'

In [43]:
# Whitespace pre-tokenizer'ın nasıl çalıştığını göster
# Her kelime ve konumu (offset) tuple olarak döner
tokenizers.pre_tokenizers.Whitespace().pre_tokenize_str("Hello, world!!!")

NameError: name 'tokenizers' is not defined

### 2.2.2 Encoding ve Decoding

In [ ]:
# Örnek bir yorum encode et
some_review = "what an awesome movie! 😊"
bpe_encoding = bpe_tokenizer.encode(some_review)
bpe_encoding  # Encoding nesnesi: ids, tokens, offsets vb. içerir

In [ ]:
# Token dizisini görüntüle (string tokenlar)
bpe_encoding.tokens

In [ ]:
# Numerik token ID'lerini görüntüle
bpe_token_ids = bpe_encoding.ids
bpe_token_ids

In [ ]:
# Vocabulary'den 'what' kelimesinin ID'sini sorgula
bpe_tokenizer.get_vocab()["what"]

In [ ]:
# Token → ID dönüşümü
bpe_tokenizer.token_to_id("what")

In [ ]:
# ID → Token dönüşümü (ters arama)
bpe_tokenizer.id_to_token(305)

In [ ]:
# ID listesini tekrar orijinal metne decode et
bpe_tokenizer.decode(bpe_token_ids)

In [ ]:
# Offsets: her token'ın orijinal metindeki başlangıç-bitiş konumları
# Hizalama (alignment) görevi için kullanışlıdır
bpe_encoding.offsets

### 2.2.3 Batch İşleme

In [ ]:
# Birden fazla metni aynı anda encode et (batch encoding)
bpe_tokenizer.encode_batch(train_reviews[:3])

In [ ]:
# Padding ve truncation ayarları:
# - pad_id=0: <pad> token'ının ID'si (özel token listesindeki ilk eleman)
# - max_length=500: 500 token'dan uzun metinler kesilir
bpe_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
bpe_tokenizer.enable_truncation(max_length=500)

In [ ]:
# Padding aktifken batch encode et
bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])
# Her encoding'den ID'leri al ve tensor'a dönüştür
bpe_batch_ids = torch.tensor([encoding.ids for encoding in bpe_encodings])
bpe_batch_ids

In [ ]:
# Attention mask: gerçek token=1, padding token=0
# Model padding token'larını yoksayması için bu mask kullanılır
attention_mask = torch.tensor([encoding.attention_mask
                               for encoding in bpe_encodings])
attention_mask

In [ ]:
# Her sekansın gerçek (padding olmayan) uzunluğunu hesapla
# attention_mask'taki 1'leri toplama yoluyla
lengths = attention_mask.sum(dim=-1)
lengths

### 2.2.4 BBPE (Byte-level BPE) Tokenization

**BBPE nedir?**  
Standart BPE'nin byte düzeyinde çalışan versiyonudur. 256 byte değerini temel birim kabul eder.

**Avantajları:**
- Emoji, aksan işaretleri, farklı dil karakterleri sorunsuz işlenir
- `<unk>` token'ına hiç gerek kalmaz (her karakter mutlaka 1-4 byte ile temsil edilebilir)
- GPT-2 ve RoBERTa bu yöntemi kullanır

**Boşluk temsili:** Boşluklar `Ġ` sembolüne, emoji ve Unicode karakterler birden fazla byte token'ına dönüştürülür.


In [20]:
# ByteLevel pre-tokenizer'ın byte dönüşümünü göster
# 😊 emojisi 4 byte'tır → 4 karakter olarak temsil edilir
# Boşluklar Ġ sembolüne dönüşür
tokenizers.pre_tokenizers.ByteLevel().pre_tokenize_str(some_review)

NameError: name 'tokenizers' is not defined

In [ ]:
# BBPE tokenizer oluştur ve eğit
bbpe_model = tokenizers.models.BPE(unk_token="<unk>")
bbpe_tokenizer = tokenizers.Tokenizer(bbpe_model)
bbpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()  # ByteLevel pre-tokenizer
bbpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000,
                                              special_tokens=special_tokens)
bbpe_tokenizer.train_from_iterator(train_reviews, bbpe_trainer)

In [ ]:
# BBPE ile encode et ve token'ları göster
bbpe_encoding = bbpe_tokenizer.encode(some_review)
bbpe_tokens = bbpe_encoding.tokens
bbpe_tokens

In [ ]:
# BBPE token ID'lerini göster
bbpe_token_ids = bbpe_encoding.ids
bbpe_token_ids

In [ ]:
# BBPE decode: Ġ sembolleri ve boşluk sorunları içerebilir
bbpe_decoded = bbpe_tokenizer.decode(bbpe_token_ids)
bbpe_decoded

In [ ]:
# Ġ sembollerini boşluğa, çift boşlukları tekli boşluğa dönüştür
bbpe_decoded.replace(" ", "").replace("Ġ", " ").strip()

### 2.2.5 WordPiece Tokenization

**WordPiece nedir?**  
Google BERT modelinde kullanılan tokenization yöntemidir. BPE'ye benzer ama birleştirme kriteri farklıdır:
- BPE: En sık geçen çift birleştirilir
- WordPiece: Dil modelinin likelihood'ını en çok artıran çift birleştirilir

**`##` ön eki:** Bir kelimenin devamı olan alt-kelimeler `##` ile işaretlenir.  
Örnek: "playing" → "play" + "##ing"


In [ ]:
# WordPiece tokenizer oluştur ve eğit
wp_model = tokenizers.models.WordPiece(unk_token="<unk>")
wp_tokenizer = tokenizers.Tokenizer(wp_model)
wp_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
wp_trainer = tokenizers.trainers.WordPieceTrainer(vocab_size=1000,
                                                  special_tokens=special_tokens)
wp_tokenizer.train_from_iterator(train_reviews, wp_trainer)

In [ ]:
# WordPiece token'larını göster
# Kelimenin devamı olan parçalar '##' öneki alır
wp_encoding = wp_tokenizer.encode(some_review)
wp_tokens = wp_encoding.tokens
wp_tokens

In [ ]:
# WordPiece token ID'leri
wp_token_ids = wp_encoding.ids
wp_token_ids

In [ ]:
# WordPiece decode
wp_decoded = wp_tokenizer.decode(wp_token_ids)
wp_decoded

In [ ]:
# ' ##' tokenlarının boşluklarını temizle ve noktalamayı düzelt
wp_decoded.replace(" ##", "").replace(" !", "!")

### 2.2.6 Unigram LM Tokenization

**Unigram LM nedir?**  
Olasılıksal bir tokenization yöntemidir. BPE ve WordPiece alt-kelime eklerken, Unigram LM çıkarır:
1. Büyük bir başlangıç vocabulary ile başlar
2. Her adımda en az bilgi kaybına yol açan token'ları çıkarır
3. Bir metin için birden fazla geçerli tokenization vardır → en olası seçilir

**Kullandığı modeller:** ALBERT, T5, XLNet, XLM-R

**SentencePiece** kütüphanesiyle de uyumludur.


In [ ]:
# Unigram LM tokenizer oluştur ve eğit
unigram_model = tokenizers.models.Unigram()
unigram_tokenizer = tokenizers.Tokenizer(unigram_model)
unigram_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
unigram_trainer = tokenizers.trainers.UnigramTrainer(
    vocab_size=1000, special_tokens=special_tokens, unk_token="<unk>")
unigram_tokenizer.train_from_iterator(train_reviews, unigram_trainer)

In [ ]:
# Unigram LM token'larını göster
unigram_encoding = unigram_tokenizer.encode(some_review)
unigram_tokens = unigram_encoding.tokens
unigram_tokens

In [ ]:
# İlk 10 token ID'sini göster
unigram_token_ids = unigram_encoding.ids[:10]
unigram_token_ids

In [ ]:
# Unigram LM decode
unigram_tokenizer.decode(unigram_token_ids)

### 2.2.7 Pretrained (Önceden Eğitilmiş) Tokenizer'lar

Kendi tokenizer'ımızı eğitmek yerine büyük modellerle birlikte gelen hazır tokenizer'ları kullanabiliriz.

**GPT-2 → BBPE kullanır**  
**BERT → WordPiece kullanır**  
**ALBERT, T5 → Unigram LM kullanır**


In [ ]:
import transformers

# GPT-2'nin pretrained BBPE tokenizer'ını yükle
gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
# 3 örnek yorumu encode et (500 token'dan uzunları kes)
gpt2_encoding = gpt2_tokenizer(train_reviews[:3], truncation=True,
                               max_length=500)

In [ ]:
# Encoding'in hangi anahtarları içerdiğini göster
gpt2_encoding.keys()

In [ ]:
# İlk yorumun ilk 10 token ID'si
gpt2_token_ids = gpt2_encoding["input_ids"][0][:10]
gpt2_token_ids

In [ ]:
# Bu token ID'lerini tekrar metne decode et
gpt2_tokenizer.decode(gpt2_token_ids)

**BERT tokenizer:** WordPiece kullanır. `[CLS]` (başlangıç) ve `[SEP]` (ayraç) özel token'larını otomatik ekler.

In [ ]:
# BERT'in pretrained WordPiece tokenizer'ını yükle
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
# padding=True → en uzun sekansa göre kısa olanları doldur
# return_tensors="pt" → PyTorch tensor'ı olarak döndür
bert_encoding = bert_tokenizer(train_reviews[:3], padding=True,
                               truncation=True, max_length=500,
                               return_tensors="pt")

In [ ]:
# Token ID matrisi: [batch_size, max_seq_len]
# [CLS] (101) ve [SEP] (102) özel tokenları otomatik eklendi
bert_encoding["input_ids"]

In [ ]:
# Attention mask: gerçek token=1, padding=0
bert_encoding["attention_mask"]

In [ ]:
# add_special_tokens=False → [CLS] ve [SEP] eklenmez
bert_encoding = bert_tokenizer(train_reviews[:3], padding=True,
                               truncation=True, max_length=500,
                               add_special_tokens=False, return_tensors="pt")
bert_encoding["input_ids"]

**ALBERT tokenizer:** Unigram LM kullanır. SentencePiece tabanlıdır; `▁` ile kelime başlangıçları işaretlenir.

In [ ]:
# ALBERT'in pretrained Unigram LM tokenizer'ını yükle
albert_tokenizer = transformers.AutoTokenizer.from_pretrained("albert-base-v2")
albert_encoding = albert_tokenizer(
    train_reviews[:3], padding=True, truncation=True, max_length=500,
    add_special_tokens=False, return_tensors="pt")
albert_token_ids = albert_encoding["input_ids"]
albert_token_ids

In [ ]:
# Kendi BPE tokenizer'ımızı HuggingFace'in PreTrainedTokenizerFast formatına sar
# Bu sayede transformers kütüphanesiyle uyumlu hale gelir
hf_tokenizer = transformers.PreTrainedTokenizerFast(
    tokenizer_object=bpe_tokenizer)
hf_encodings = hf_tokenizer(train_reviews[:3], padding=True, truncation=True,
                            max_length=500, return_tensors="pt")
hf_encodings["input_ids"]

## 2.3 Sentiment Analysis Modelini Oluşturma ve Eğitme

### `collate_fn` – Özelleştirilmiş Batch Hazırlama

IMDB veri seti dictionary nesneleri içerdiğinden, DataLoader'ın batchleri nasıl oluşturacağını özelleştirmemiz gerekir:
- Her örnekten metin ve etiketi ayrıştır
- Metinleri tokenize et ve padding yap
- Etiketleri tensor'a dönüştür


In [ ]:
def collate_fn(batch, tokenizer=bert_tokenizer):
    """
    DataLoader için özelleştirilmiş batch hazırlama fonksiyonu.
    
    Her batch: liste halinde dataset örnekleri (dictionary'ler)
    Çıktı: (encodings_dict, labels_tensor) çifti
    """
    reviews = [review["text"] for review in batch]           # Metinleri al
    labels  = [[review["label"]] for review in batch]        # Etiketleri al
    # Tokenize et, padding yap, tensor'a dönüştür
    encodings = tokenizer(reviews, padding=True, truncation=True,
                          max_length=200, return_tensors="pt")
    labels = torch.tensor(labels, dtype=torch.float32)       # Float (BCELoss için)
    return encodings, labels

batch_size = 256
imdb_train_loader = DataLoader(imdb_train_set, batch_size=batch_size,
                               collate_fn=collate_fn, shuffle=True)
imdb_valid_loader = DataLoader(imdb_valid_set, batch_size=batch_size,
                               collate_fn=collate_fn)
imdb_test_loader  = DataLoader(imdb_test_set,  batch_size=batch_size,
                               collate_fn=collate_fn)

**Basit GRU tabanlı Sentiment Analysis Modeli:**

Mimari:
1. **Embedding** → token ID'leri yoğun vektörlere
2. **GRU** → sekans üzerinde bağlamı öğren
3. **Linear** → son hidden state'ten binary sınıflandırma çıktısı

**Not:** `padding_idx=pad_id` → Embedding katmanı padding token'ını sıfır vektörü olarak işler.


In [ ]:
class SentimentAnalysisModel(nn.Module):
    """
    GRU tabanlı binary sentiment sınıflandırıcı.
    
    Girdi: BERT tokenizer'ın ürettiği encoding dictionary
           {input_ids: (B, L), attention_mask: (B, L)}
    Çıktı: (B, 1) → ham logit (BCEWithLogitsLoss ile kullanılır)
    """
    def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64,
                 pad_id=0, dropout=0.2):
        super().__init__()
        # padding_idx: bu ID'ye sahip tokenlar için gradient hesaplanmaz
        self.embed = nn.Embedding(vocab_size, embed_dim,
                                  padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)  # Binary sınıflandırma: 1 çıkış

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])  # Token ID → embedding
        _outputs, hidden_states = self.gru(embeddings)   # Son hidden state al
        # hidden_states[-1]: son GRU katmanının çıktısı → (batch, hidden_dim)
        return self.output(hidden_states[-1])             # Logit üret

### Pack/Pad Mekanizması – Değişken Uzunluklu Sekanslar

**Sorun:** Bir batch'te farklı uzunluklu sekanslar vardır. Kısa olanlar padding token'ları ile doldurulur. Ancak RNN bu padding token'larını da işler → gereksiz hesaplama ve bozulan hidden state.

**Çözüm:** `pack_padded_sequence` / `pad_packed_sequence`
- `pack_padded_sequence`: Padding'leri atlayarak sadece gerçek token'ları sıkıştırılmış formda RNN'e verir
- `pad_packed_sequence`: Sıkıştırılmış çıktıyı padding ekleyerek normal tensor'a geri dönüştürür

**`enforce_sorted=False`:** Uzunluğa göre sıralama zorunluluğunu kaldırır.


In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# Örnek: iki sekans, biri 2 gerçek token (0=padding), biri 4 gerçek token
sequences = torch.tensor([[1, 2, 0, 0], [5, 6, 7, 8]])
# lengths=(2, 4) → her sekansın gerçek uzunluğunu belirt
packed = pack_padded_sequence(sequences, lengths=(2, 4),
                              enforce_sorted=False, batch_first=True)
# PackedSequence: data (sadece gerçek tokenlar), batch_sizes, sorted_indices
packed

In [ ]:
# Sıkıştırılmış sekansı padding ekleyerek geri aç
padded, lengths = pad_packed_sequence(packed, batch_first=True)
padded, lengths

**Packed Sequence kullanan Sentiment Model:**

In [ ]:
class SentimentAnalysisModelPackedSeq(nn.Module):
    """
    Pack/Pad mekanizması kullanan geliştirilmiş sentiment modeli.
    
    Padding token'ları RNN'e iletilmez → daha doğru hidden state,
    daha az hesaplama ve bellek kullanımı.
    """
    def __init__(self, vocab_size, n_layers=2, embed_dim=128,
                 hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim,
                                  padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        # attention_mask'ı kullanarak her sekansın gerçek uzunluğunu hesapla
        lengths = encodings["attention_mask"].sum(dim=1)                      # Gerçek uzunluklar
        # Embedding'leri sıkıştır: padding token'ları RNN'e gitmez
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),      # CPU'da olmalı
                                      batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)                            # Packed girdi
        return self.output(hidden_states[-1])

In [ ]:
torch.manual_seed(42)

vocab_size = bert_tokenizer.vocab_size  # BERT vocabulary boyutu (~30.522)
imdb_model_ps = SentimentAnalysisModelPackedSeq(vocab_size).to(device)

n_epochs = 10
# BCEWithLogitsLoss: Binary Cross-Entropy + Sigmoid birleşimi
# Numerik olarak daha kararlı (sigmoid ayrı uygulamaktan daha iyi)
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_ps.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)  # Binary accuracy

history = train(imdb_model_ps, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

### Bidirectional RNN (Çift Yönlü RNN)

**Neden bidirectional?**  
Standart RNN soldaki bağlamı kullanır. Oysa "The movie was **not** bad at all" cümlesinde "not" kelimesini anlamak için hem önceki hem sonraki bağlam gereklidir.

**Bidirectional GRU:**
- **Forward pass**: Soldan sağa okur
- **Backward pass**: Sağdan sola okur
- Her zaman adımında iki hidden state birleştirilir (concatenate)
- Son hidden state boyutu: `2 × hidden_dim`

**Kod değişiklikleri:**
- `bidirectional=True`
- `output` katmanı `2 * hidden_dim` girdi alır
- Son hidden state: `[-2:]` (son 2 katman: forward'ın sonu + backward'ın sonu)


In [ ]:
class SentimentAnalysisModelBidi(nn.Module):
    """
    Bidirectional GRU tabanlı sentiment modeli.
    
    Hem soldan sağa hem sağdan sola bağlamı kullanır.
    Son hidden state: forward son katman + backward son katman (concatenate)
    """
    def __init__(self, vocab_size, n_layers=2, embed_dim=128,
                 hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim,
                                  padding_idx=pad_id)
        # bidirectional=True → çift yönlü GRU
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)
        # 2 yön × hidden_dim → birleşik temsil
        self.output = nn.Linear(2 * hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        n_dims = self.output.in_features  # 2 * hidden_dim
        # hidden_states shape: (n_layers * 2, batch, hidden_dim)
        # [-2:] → son 2 katman (forward + backward son katmanlar)
        # permute + reshape → (batch, 2 * hidden_dim)
        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
        return self.output(top_states)

In [ ]:
torch.manual_seed(42)

vocab_size = bert_tokenizer.vocab_size
imdb_model_bidi = SentimentAnalysisModelBidi(vocab_size).to(device)

n_epochs = 10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bidi.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(imdb_model_bidi, optimizer, xentropy, accuracy, imdb_train_loader,
                imdb_valid_loader, n_epochs)

GPU RAM temizleme:

In [ ]:
Out.clear()  # Jupyter Out dictionary'sini temizle
del_vars(["albert_token_ids", "attention_mask", "bpe_batch_ids", "encoded_text",
          "lengths", "optimizer", "padded", "probs", "samples", "sequences",
          "x", "xentropy", "y", "Y_logits"])

In [ ]:
# BERT modelini yükle (sadece embedding'leri kullanmak için)
bert_model = transformers.AutoModel.from_pretrained("bert-base-uncased")
# word_embeddings: 30522 token × 768 boyutlu embedding matrisi
bert_model.embeddings.word_embeddings

In [ ]:
class SentimentAnalysisModelPreEmbeds(nn.Module):
    """
    BERT'in önceden eğitilmiş embedding'lerini kullanan sentiment modeli.
    
    Embedding ağırlıkları dondurulur (freeze=True) → eğitim sırasında güncellenmez.
    Sadece GRU ve output katmanı eğitilir.
    """
    def __init__(self, pretrained_embeddings, n_layers=2, hidden_dim=64,
                 dropout=0.2):
        super().__init__()
        # BERT'in ağırlıklarını kopyala ve dondur
        weights = pretrained_embeddings.weight.data
        self.embed = nn.Embedding.from_pretrained(weights, freeze=True)  # freeze=True!
        embed_dim = weights.shape[-1]  # BERT için 768
        # Bidirectional GRU
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)
        self.output = nn.Linear(2 * hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        n_dims = self.output.in_features
        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
        return self.output(top_states)

In [ ]:
torch.manual_seed(42)

# BERT'in word embedding katmanını modele geç
imdb_model_bert_embeds = SentimentAnalysisModelPreEmbeds(
    bert_model.embeddings.word_embeddings).to(device)

n_epochs = 10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert_embeds.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(imdb_model_bert_embeds, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

**BERT'in contextualized output'larını anlama:**

BERT iki tür çıktı üretir:
- `last_hidden_state`: Tüm token pozisyonları için bağlamsal temsil → shape: (batch, seq_len, 768)
- `pooler_output`: `[CLS]` token'ının dense katmandan geçirilmiş hali → shape: (batch, 768)


In [ ]:
# BERT'in tam modelini çalıştırarak çıktı shape'lerini incele
bert_encoding = bert_tokenizer(train_reviews[:3], padding=True,
                               max_length=200, truncation=True,
                               return_tensors="pt")
bert_output = bert_model(**bert_encoding)
# last_hidden_state: her token için 768 boyutlu bağlamsal temsil
bert_output.last_hidden_state.shape  # (3, seq_len, 768)

GPU RAM temizleme:

In [ ]:
del_vars(["bert_model"])

**BERT contextualized embeddings + GRU:**

Bu yaklaşımda BERT tüm sekans için bağlamsal embedding üretir (her token için ayrı), sonra bu embedding'ler GRU'ya beslenir.

**BERT dondurulur (`requires_grad_(False)`)**: BERT'in ~110 milyon parametresini güncellemek çok maliyetli. Sadece GRU eğitilir.


In [ ]:
class SentimentAnalysisModelBert(nn.Module):
    """
    BERT'in contextualized output'larını GRU'ya besleyen model.
    
    Akış: Token ID → BERT (frozen) → bağlamsal embedding → GRU → linear
    """
    def __init__(self, n_layers=2, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        embed_dim = self.bert.config.hidden_size  # 768
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        # BERT'i çalıştır → tüm token'lar için bağlamsal temsil al
        contextualized_embeddings = self.bert(**encodings).last_hidden_state
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(contextualized_embeddings, lengths=lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        return self.output(hidden_states[-1])

In [ ]:
torch.manual_seed(42)

imdb_model_bert = SentimentAnalysisModelBert().to(device)
# BERT'in tüm parametrelerini dondur (sadece GRU ve output eğitilecek)
imdb_model_bert.bert.requires_grad_(False)

n_epochs = 4  # Frozen BERT ile daha az epoch yeterli
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(imdb_model_bert, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

GPU RAM temizleme:

In [ ]:
del_vars(["imdb_model_bert"])

In [ ]:
class SentimentAnalysisModelBert2(nn.Module):
    """
    [CLS] token'ının last_hidden_state çıktısını doğrudan kullanan model.
    GRU'ya gerek yok: [CLS] zaten tüm sekansı özetleyen özel bir token.
    """
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size, 1)  # 768 → 1

    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        # [:, 0] → her örnekte [CLS] token'ının temsili (indeks 0)
        return self.output(bert_output.last_hidden_state[:, 0])

**pooler_output kullanan model (Bert3):**

`pooler_output` = `[CLS]` token'ının `tanh` aktivasyonlu bir Linear katmandan geçirilmiş hali.  
BERT orijinal olarak NSP (Next Sentence Prediction) görevi için bu çıktıyı kullanır.


In [ ]:
class SentimentAnalysisModelBert3(nn.Module):
    """
    BERT'in pooler_output'unu kullanan sınıflandırıcı.
    pooler_output: [CLS] token → Linear(768, 768) + tanh
    """
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        return self.output(bert_output.pooler_output)  # pooler_output kullan

In [ ]:
torch.manual_seed(42)

imdb_model_bert3 = SentimentAnalysisModelBert3().to(device)
imdb_model_bert3.bert.requires_grad_(False)  # BERT'i dondur

n_epochs = 5
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert3.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

# Aşama 1: Sadece output katmanını eğit
history = train(imdb_model_bert3, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

In [ ]:
# Aşama 2: BERT'in pooler katmanını serbest bırak (partial fine-tuning)
# Pooler NSP görevi için eğitildi; sentiment için de uyarlanabilir
imdb_model_bert3.bert.pooler.requires_grad_(True)

history = train(imdb_model_bert3, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

GPU RAM temizleme:

In [ ]:
del_vars(["imdb_model_bert3"])

# 3. Task-Specific Classes (Göreve Özel Sınıflar)

Hugging Face Transformers kütüphanesi, yaygın NLP görevleri için hazır model sınıfları sağlar:

| Sınıf | Görev |
|-------|-------|
| `BertForSequenceClassification` | Metin sınıflandırma |
| `BertForTokenClassification` | NER, POS tagging |
| `BertForQuestionAnswering` | Soru-cevap |
| `BertForMaskedLM` | Masked language modeling |

Bu sınıflar BERT'i alıp üstüne görev başlığı (task head) ekler.  
`num_labels=2` → İkili sınıflandırma (pozitif / negatif).


In [ ]:
from transformers import BertForSequenceClassification

torch.manual_seed(42)
# BERT + Sequence Classification head
# dtype=torch.float16 → yarı hassasiyetli (GPU belleğinden tasarruf)
bert_for_binary_clf = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2, dtype=torch.float16).to(device)

In [ ]:
# Bir örnek cümleyi sınıflandır
encoding = bert_tokenizer(["This was a great movie!"])
with torch.no_grad():
    output = bert_for_binary_clf(
        input_ids=torch.tensor(encoding["input_ids"], device=device),
        attention_mask=torch.tensor(encoding["attention_mask"], device=device))

# output.logits: 2 sınıf için ham skorlar (negatif, pozitif)
output.logits

In [ ]:
# Logits'i olasılıklara dönüştür (softmax)
# İlk değer: negatif olasılığı, ikinci değer: pozitif olasılığı
torch.softmax(output.logits, dim=-1)

In [ ]:
# labels parametresi verilince model doğrudan loss'u da hesaplar
with torch.no_grad():
    output = bert_for_binary_clf(
        input_ids=torch.tensor(encoding["input_ids"], device=device),
        attention_mask=torch.tensor(encoding["attention_mask"], device=device),
        labels=torch.tensor([1], device=device))  # 1 = pozitif

# output.loss: cross-entropy loss değeri
output.loss

Her Transformers modeli `.config` attribute'u ile yapılandırma bilgilerini tutar:

In [ ]:
# Modelin tüm yapılandırma parametrelerini göster
# hidden_size, num_attention_heads, num_hidden_layers vb.
bert_for_binary_clf.config

# 4. Trainer API

**Trainer API nedir?**  
Hugging Face'in yüksek seviyeli eğitim döngüsü. Manuel `train()` fonksiyonu yazmak yerine kullanılabilir.

**Avantajları:**
- Distributed training (çoklu GPU) desteği
- Mixed precision training (fp16)
- Gradient accumulation
- Automatic checkpoint saving
- Weights & Biases / TensorBoard entegrasyonu

**Gereksinim:** Trainer API önceden tokenize edilmiş veri setleri bekler.


In [ ]:
 def tokenize_batch(batch):
    """Dataset.map() için tokenization fonksiyonu."""
    return bert_tokenizer(batch["text"], truncation=True, max_length=200)

# Tüm veri setini toplu olarak tokenize et (batched=True → daha hızlı)
tok_imdb_train_set = imdb_train_set.map(tokenize_batch, batched=True)
tok_imdb_valid_set = imdb_valid_set.map(tokenize_batch, batched=True)
tok_imdb_test_set  = imdb_test_set.map(tokenize_batch, batched=True)

Trainer API streaming metrikleri desteklemez; kendi değerlendirme fonksiyonumuzu yazarız:

In [ ]:
def compute_accuracy(pred):
    """
    Trainer API için custom evaluation metriği.
    pred.predictions: model çıktısı (logits array)
    pred.label_ids: gerçek etiketler
    """
    return {"accuracy": (pred.label_ids == pred.predictions.argmax(-1)).mean()}

In [ ]:
from transformers import TrainingArguments

# Eğitim argümanlarını tanımla
train_args = TrainingArguments(
    output_dir="my_imdb_model",              # Checkpoint kayıt dizini
    num_train_epochs=2,                       # 2 epoch
    per_device_train_batch_size=128,          # Eğitim batch boyutu
    per_device_eval_batch_size=128,           # Değerlendirme batch boyutu
    eval_strategy="epoch",                    # Her epoch sonunda değerlendir
    logging_strategy="epoch",                 # Her epoch sonunda logla
    save_strategy="epoch",                    # Her epoch sonunda checkpoint kaydet
    load_best_model_at_end=True,              # En iyi modeli yükle
    metric_for_best_model="accuracy",         # En iyiyi accuracy'e göre seç
    report_to="none")                         # W&B / TensorBoard raporlama kapalı

In [ ]:
from transformers import DataCollatorWithPadding, Trainer

# DataCollatorWithPadding: batch içinde dinamik padding yapar
trainer = Trainer(
    bert_for_binary_clf, train_args,
    train_dataset=tok_imdb_train_set,
    eval_dataset=tok_imdb_valid_set,
    compute_metrics=compute_accuracy,
    data_collator=DataCollatorWithPadding(bert_tokenizer))  # Otomatik padding

train_output = trainer.train()

GPU RAM temizleme:

In [ ]:
del_vars(["bert_for_binary_clf"])

# 5. Pipelines (Boru Hatları)

**Pipeline nedir?**  
Hugging Face `pipeline` API'si, pretrained bir modeli birkaç satırla kullanmaya izin veren en yüksek seviyeli arayüzdür.

**Avantajları:**
- Tokenization, inference ve decode otomatik
- Modeli Hub'dan direkt indir ve kullan
- Production-ready inference için ideal

**`distilbert-base-uncased-finetuned-sst-2-english`**: BERT'in distilled (damıtılmış) versiyonu, SST-2 sentiment veri seti üzerinde fine-tune edilmiş. %40 daha hızlı, ~%97 aynı performans.


In [ ]:
from transformers import pipeline

# Sentiment analysis pipeline: model Hub'dan otomatik indirilir
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
classifier_imdb = pipeline("sentiment-analysis", model=model_name,
                           truncation=True, max_length=512)
# İlk 10 yorumu sınıflandır
classifier_imdb(train_reviews[:10])

In [ ]:
# Pipeline'ın tüm validation seti üzerindeki accuracy'sini hesapla
accuracy = torchmetrics.Accuracy(task="binary").to(device)
with torch.no_grad():
    text_imdb_valid_loader = DataLoader(imdb_valid_set, batch_size=256)
    for index, batch in enumerate(text_imdb_valid_loader):
        # Pipeline her metin için {"label": "POSITIVE"/"NEGATIVE", "score": ...} döner
        y_pred = classifier_imdb(batch["text"], truncation=True)
        y_pred = torch.tensor([pred["label"] == "POSITIVE" for pred in y_pred], dtype=int)
        accuracy.update(y_pred, batch["label"])
        print(f"\r{index + 1}/{len(text_imdb_valid_loader)}", end="")

accuracy.compute()

**⚠️ Model Bias (Model Yanlılığı) Uyarısı:**

Binary sınıflandırma modelleri coğrafi, kültürel veya demografik yanlılıklar içerebilir.  
Bu örnekte belirli ülkelerden bahseden nötr cümleler POSITIVE ya da NEGATIVE sınıflandırılabilir.

**Çözüm:** Nötr sınıf içeren modeller kullanmak yanlılığı azaltabilir.


In [ ]:
# Binary sınıflandırmanın model yanlılığını nasıl güçlendirebileceğini göster
countries = ["Iraq", "Thailand", "the USA", "Vietnam"]
texts = [f"I am from {country}" for country in countries]
# "I am from X" → nötr bir cümle, ama model bazı ülkeler için yanlı davranabilir
list(zip(countries, classifier_imdb(texts)))

In [ ]:
# Nötr sınıflı model yanlılığı azaltır
# Not: uyarı normaldir – bu modelin pooler ağırlıkları sınıflandırma için kullanılmaz
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
classifier_imdb_with_neutral = pipeline("sentiment-analysis", model=model_name)
list(zip(countries, classifier_imdb_with_neutral(texts)))

In [ ]:
# NLI (Natural Language Inference) görevi için pipeline
# Girdi: "öncül [SEP] hipotez [SEP]" formatında cümle çifti
# Çıktı: entailment (çıkarım), neutral, contradiction sınıflandırması
model_name = "huggingface/distilbert-base-uncased-finetuned-mnli"
classifier_mnli = pipeline("text-classification", model=model_name)
classifier_mnli([
    "She loves me. [SEP] She loves me not. [SEP]",       # contradiction
    "Alice just woke up. [SEP] Alice is awake. [SEP]",   # entailment
    "I like dogs. [SEP] Everyone likes dogs. [SEP]"])    # neutral

GPU RAM temizleme:

In [ ]:
Out.clear()
del_vars(["classifier_imdb", "classifier_mnli", "classifier_imdb_with_neutral",
          "trainer"])

# 6. Encoder–Decoder Ağı ile Neural Machine Translation (NMT)

**Neural Machine Translation (Sinir Ağı Tabanlı Makine Çevirisi):**  
Bir dildeki metni başka bir dile otomatik çeviren deep learning sistemi.

**Encoder–Decoder Mimarisi:**
```
Kaynak metin (İngilizce)
        ↓
   [ENCODER]  → Anlam vektörü (context/thought vector)
        ↓
   [DECODER]  ← Hedef metin (İspanyolca) — eğitimde teacher forcing
        ↓
  Çeviri çıktısı
```

**Encoder:** Kaynak dilin tüm anlam bilgisini sıkıştırır → final hidden state  
**Decoder:** Bu hidden state'ten başlayarak hedef dili kelime kelime üretir

**Tatoeba MT veri seti:** Çok dilli cümle çiftlerinden oluşan açık kaynak veri seti.  
Burada **İngilizce → İspanyolca** çeviri yapacağız.


In [ ]:
# Notebook'u bu bölümden başlatmak isteyenler için bağımlılıkları tekrar yükle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
from datasets import load_dataset
import tokenizers

In [ ]:
# Tatoeba veri setini yükle (İngilizce → İspanyolca)
# validation setini train/test olarak böl (train_size=0.8)
nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa",
    split=["validation", "test"])
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)
nmt_train_set, nmt_valid_set = split["train"], split["test"]

In [ ]:
# Veri setindeki ilk örneği göster
# source_text: İngilizce, target_text: İspanyolca
nmt_train_set[0]

**NMT için BPE Tokenizer eğitme:**

NMT modelinde kaynak ve hedef dil için **aynı tokenizer** kullanılır. Bu paylaşılan vocabulary (shared vocabulary) yaklaşımı:
- Çapraz dil kelime eşleşmelerini öğrenmeye yardımcı olur
- İki dildeki ortak yapıları (özel isimler, sayılar) aynı token ID'siyle temsil eder

**Özel tokenlar:**
- `<pad>` (ID=0): Padding
- `<unk>` (ID=1): Bilinmeyen token
- `<s>` (ID=2): Başlangıç tokeni (BOS – Beginning Of Sentence)
- `</s>` (ID=3): Bitiş tokeni (EOS – End Of Sentence)


In [ ]:
def train_eng_spa():
    """Eğitim veri setindeki tüm metin çiftlerini yield eden generator."""
    for pair in nmt_train_set:
        yield pair["source_text"]  # İngilizce
        yield pair["target_text"]  # İspanyolca

max_length = 256  # Maksimum token uzunluğu
vocab_size = 10_000  # Vocabulary boyutu

# BPE tokenizer oluştur
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()

# Özel tokenlar: <pad>, <unk>, <s> (BOS), </s> (EOS)
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>", "</s>"])

# Hem İngilizce hem İspanyolca metin üzerinde eğit → paylaşılan vocabulary
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)

In [ ]:
# İngilizce cümleyi encode et
nmt_tokenizer.encode("I like soccer").ids

In [ ]:
# İspanyolca cümleyi <s> BOS tokeni ile encode et
# <s> (ID=2) → decoder'ın başlangıç tokeni
nmt_tokenizer.encode("<s> Me gusta el fútbol").ids

**`NmtPair` – Veri namedtuple'ı:**

Encoder ve decoder için kaynak/hedef veri çiftlerini organize eder.  
`.to(device)` methodu tüm tensor'ları GPU'ya taşır.


In [ ]:
from collections import namedtuple

# Encoder-Decoder modeli için gerekli 4 tensor'ı bir arada tutan yapı
fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        """Tüm tensor'ları belirtilen cihaza (GPU/CPU) taşı."""
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))

**`nmt_collate_fn` – NMT Batch Hazırlama:**

**Teacher Forcing:** Eğitimde decoder'a gerçek hedef token'ları verilir (modelin kendi tahminleri değil). Bu eğitimi kararlı ve hızlı yapar.

**Girdi / Hedef kayması (offset):**
- Decoder girdisi: `<s> benim adım` (başlangıç tokeni + hedef)
- Hedef etiketler: `benim adım </s>` (hedef + bitiş tokeni)
- `tgt_token_ids[:, :-1]` → son token atılır (decoder girdisi)
- `tgt_token_ids[:, 1:]` → ilk token atılır (tahmin edilecek etiket)


In [ ]:
def nmt_collate_fn(batch):
    """
    NMT modeli için batch hazırlama fonksiyonu.
    
    Teacher Forcing için:
    - Decoder girdisi: '<s> hedef_metin' (son token hariç)
    - Etiket: 'hedef_metin </s>' (ilk token hariç)
    
    Örnek:
    Hedef: '<s> Me gusta el fútbol </s>'
    Girdi: '<s> Me gusta el fútbol'   (son </s> yok)
    Etiket: 'Me gusta el fútbol </s>'  (ilk <s> yok)
    """
    src_texts = [pair['source_text'] for pair in batch]
    # Hedef metni <s> ile başlat, </s> ile bitir
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    
    # Decoder girdisi: [:-1] → son token atılır
    # Etiketler: [1:] → ilk token (<s>) atılır
    inputs = NmtPair(src_token_ids, src_mask,
                     tgt_token_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

batch_size = 32
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn)
nmt_test_loader  = DataLoader(nmt_test_set,  batch_size=batch_size,
                              collate_fn=nmt_collate_fn)

**NMT Model Mimarisi:**

```
Kaynak (İngilizce): "I like soccer"
        ↓
    Embedding
        ↓
    Encoder GRU  → final hidden_state (anlam vektörü)
                           ↓
Hedef (İspanyolca): "<s> Me gusta..."   
        ↓
    Embedding
        ↓
    Decoder GRU ← initial hidden_state (encoder'dan gelen)
        ↓
    Linear → vocab_size logits
        ↓
    "Me gusta el fútbol </s>"
```


In [ ]:
class NmtModel(nn.Module):
    """
    Basit Encoder–Decoder NMT modeli (attention mekanizması olmadan).
    
    Encoder: Kaynak dilin anlamını hidden state'e sıkıştırır.
    Decoder: Encoder'ın hidden state'inden başlayarak hedef dili üretir.
    
    Sınırlama: Uzun cümleler için encoder tek bir vektöre çok fazla bilgi
    sıkıştırmalı → information bottleneck (bilgi darboğazı) sorunu.
    """
    def __init__(self, vocab_size, embed_dim=512, pad_id=0, hidden_dim=512,
                 n_layers=2):
        super().__init__()
        # Paylaşılan embedding: kaynak ve hedef dil için aynı vocabulary
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        # Encoder: kaynak dilin bağlamını öğrenir
        self.encoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                              batch_first=True)
        # Decoder: hedef dili token token üretir
        self.decoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                              batch_first=True)
        # Çıktı katmanı: her token için vocab_size logit
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)  # Kaynak embedding
        tgt_embeddings = self.embed(pair.tgt_token_ids)  # Hedef embedding
        src_lengths = pair.src_mask.sum(dim=1)
        # Encoder: pack → GRU → sadece final hidden state gerekli
        src_packed = pack_padded_sequence(
            src_embeddings, lengths=src_lengths.cpu(),
            batch_first=True, enforce_sorted=False)
        _, hidden_states = self.encoder(src_packed)  # _ = outputs, ihtiyacımız yok
        # Decoder: encoder'ın hidden state'inden başla
        outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        return self.output(outputs).permute(0, 2, 1)  # (B, vocab_size, seq_len)

torch.manual_seed(42)
vocab_size = nmt_tokenizer.get_vocab_size()
nmt_model = NmtModel(vocab_size).to(device)

In [ ]:
n_epochs = 10
# CrossEntropyLoss: ignore_index=0 → padding token'ı için loss hesaplanmaz
xentropy = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.NAdam(nmt_model.parameters(), lr=0.001)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size)
accuracy = accuracy.to(device)

history = train(nmt_model, optimizer, xentropy, accuracy,
                nmt_train_loader, nmt_valid_loader, n_epochs)

In [ ]:
# Modeli kaydet
torch.save(nmt_model.state_dict(), "my_nmt_model.pt")

**`translate()` – Greedy Decoding:**

Inference sırasında decoder adım adım token üretir:
1. Boş hedef metninden başla
2. Her adımda tüm sekansı modele ver
3. En yüksek logitli token'ı seç (argmax = greedy)
4. Bu token'ı metne ekle
5. EOS token (`</s>`) görülene kadar tekrar et


In [ ]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    """
    Greedy decoding ile kaynak metni hedef dile çevirir.
    
    Greedy Decoding: Her adımda en yüksek olasılıklı token seçilir.
    Basit ama optimal değil — Beam Search daha iyi sonuçlar verir.
    
    Args:
        model: Eğitilmiş NmtModel
        src_text: Çevrilecek kaynak metin (İngilizce)
        max_length: Maksimum çıktı uzunluğu
        pad_id: Padding token ID (0)
        eos_id: End-of-sequence token ID (3 = </s>)
    
    Returns:
        str: Çevrilmiş metin (İspanyolca)
    """
    tgt_text = ""
    token_ids = []
    for index in range(max_length):
        # Her adımda güncel çeviriye göre batch oluştur
        batch, _ = nmt_collate_fn([{"source_text": src_text,
                                    "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)  # Greedy: en yüksek logit
            next_token_id = Y_token_ids[0, index]  # Şu anki adımın tokeni

        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:  # </s> tokeni görüldü → dur
            break
    return tgt_text

In [ ]:
nmt_model.eval()
# Kısa cümle testi
translate(nmt_model, "I like soccer.")

In [ ]:
# Daha uzun cümle testi
longer_text = "I like to play soccer with my friends."
translate(nmt_model, longer_text)

## 6.1 Beam Search

**Beam Search nedir?**  
Greedy decoding'in geliştirilmiş versiyonu. Tek bir en iyi yolu değil, **k en umut verici çeviriyi** aynı anda takip eder.

**Algoritma:**
1. `k` (beam width) adet en olası başlangıç tokeni ile başla
2. Her adımda her aday çeviriyi tüm vocab ile genişlet: `k × vocab_size` kombinasyon
3. Bu kombinasyonlar arasından en yüksek olasılıklı `k` tanesini tut
4. Tüm `k` çeviri EOS ile bitene kadar tekrar et
5. En yüksek skorlu çeviriyi döndür

**Log probability:** Çarpım yerine toplama kullanılır (underflow riski azalır):  
`log(p(S, W)) = log(p(S)) + log(p(W|S))`

**Length penalty:** Uzun çeviriler daha düşük olasılık alır. Bu dezavantajı dengelemek için uzunluk cezası uygulanır:  
`score = log_proba / ((5 + length)^α / 6^α)` — α genellikle 0.6


----

Bu, beam search’ün oldukça temel bir implementasyonudur. Okunabilir ve anlaşılır olması için yazmaya çalıştım, ancak hız açısından kesinlikle optimize edilmemiştir.

Fonksiyon önce modeli kullanarak çeviriyi başlatmak için en iyi k kelimeyi bulur (burada k, beam width’tür). Ardından bu top k translations’ın her biri için, bu çeviriye eklenebilecek tüm olası kelimelerin conditional probabilities’ini değerlendirir.

Bu genişletilmiş translations ve onların olasılıkları (probabilities) adaylar (candidates) listesine eklenir. Tüm top k translations ve onları tamamlayabilecek tüm kelimeler üzerinden geçildikten sonra, en yüksek olasılığa sahip top k candidates seçilir.

Bu işlem, tüm aday çeviriler EOS token ile bitene kadar tekrar tekrar yapılır. Sonunda en iyi translation, EOS token kaldırıldıktan sonra döndürülür.

⸻

Not:
Eğer p(S), S cümlesinin olasılığıysa ve p(W|S), çevirinin S ile başladığı durumda kelime W’nun conditional probability’si ise,

S′ = concat(S, W) cümlesinin olasılığı şu şekilde hesaplanır:

p(S') = p(S) * p(W|S)

Daha fazla kelime eklendikçe bu olasılık giderek küçülür. Bu değerin çok küçük olup floating point precision errors oluşturmasını önlemek için fonksiyon probabilities yerine log probabilities kullanır.

Hatırlarsak:

log(a*b) = log(a) + log(b)

Dolayısıyla:

log(p(S')) = log(p(S)) + log(p(W|S))

In [ ]:
def beam_search(model, src_text, beam_width=3, max_length=20,
                verbose=False, length_penalty=0.6):
    """
    Beam Search ile çeviri üretir.
    
    Args:
        model: Eğitilmiş NMT modeli
        src_text: Kaynak metin
        beam_width (k): Aynı anda takip edilen çeviri sayısı
                        Yüksek k → daha iyi kalite, daha yavaş
        max_length: Maksimum çeviri uzunluğu
        verbose: Her adımda top-k çevirileri yazdır
        length_penalty: Uzun çevirileri penalize eden alfa parametresi
    
    Returns:
        str: En yüksek skorlu çeviri
    """
    top_translations = [(torch.tensor(0.), "")]  # (log_proba, metin) çifti
    
    for index in range(max_length):
        if verbose:
            print(f"Top {beam_width} translations so far:")
            for log_proba, tgt_text in top_translations:
                print(f"    {log_proba.item():.3f} – {tgt_text}")

        candidates = []
        for log_proba, tgt_text in top_translations:
            if tgt_text.endswith(" </s>"):
                # EOS geldi → bu çeviri tamamlandı, olduğu gibi ekle
                candidates.append((log_proba, tgt_text))
                continue
            
            # Mevcut çeviri için sonraki olası token'ları hesapla
            batch, _ = nmt_collate_fn([{"source_text": src_text,
                                        "target_text": tgt_text}])
            with torch.no_grad():
                Y_logits = model(batch.to(device))
                # Log softmax: log olasılıkları hesapla (numerik kararlılık için)
                Y_log_proba = F.log_softmax(Y_logits, dim=1)
                # Şu anki adım için en yüksek k olasılıklı token'ı bul
                Y_top_log_probas = torch.topk(Y_log_proba, k=beam_width, dim=1)

            for beam_index in range(beam_width):
                next_token_log_proba = Y_top_log_probas.values[0, beam_index, index]
                next_token_id = Y_top_log_probas.indices[0, beam_index, index]
                next_token = nmt_tokenizer.id_to_token(next_token_id)
                next_tgt_text = tgt_text + " " + next_token
                # log(P(S, W)) = log(P(S)) + log(P(W|S))
                candidates.append((log_proba + next_token_log_proba, next_tgt_text))

        def length_penalized_score(candidate, alpha=length_penalty):
            """Uzunluk cezası uygulanmış skor: uzun çevirilerin dezavantajını dengeler."""
            log_proba, text = candidate
            length = len(text.split())
            penalty = ((5 + length) ** alpha) / (6 ** alpha)
            return log_proba / penalty

        # Tüm adaylar arasından en iyi k tanesini seç
        top_translations = sorted(candidates,
                                  key=length_penalized_score,
                                  reverse=True)[:beam_width]

    # En yüksek skorlu çeviriyi döndür
    return top_translations[-1][1]

In [ ]:
# Beam Search ile orta uzunlukta cümleyi çevir (beam_width=3)
beam_search(nmt_model, longer_text, beam_width=3)

In [ ]:
# Daha uzun cümle testi
longest_text = "I like to play soccer with my friends at the beach."
beam_search(nmt_model, longest_text, beam_width=3)

GPU RAM temizleme:

In [ ]:
del_vars(["nmt_model"])

# 7. Attention Mechanisms (Dikkat Mekanizmaları)

**Attention Mekanizmasının Motivasyonu:**

Encoder–Decoder modelinde **information bottleneck** (bilgi darboğazı) sorunu vardır:
- Encoder tüm kaynak cümleyi **tek bir sabit boyutlu vektöre** (final hidden state) sıkıştırmalı
- Uzun cümleler için bu vektör çok fazla bilgi tutmak zorunda → kalite düşer

**Attention çözümü:**
- Decoder her adımda encoder'ın **tüm** çıktılarına erişir (sadece son hidden state değil)
- Her decoder adımı için hangi encoder çıktısına ne kadar "dikkat" edilmesi gerektiği öğrenilir
- Bu "dikkat ağırlıkları" softmax ile normalize edilir → yorumlanabilir hizalama haritası

### Luong Attention (Dot-Product Attention)

En basit attention türü:

```
Score(query, key) = query · keyᵀ
Weight = softmax(Score)
Output = Weight × Value
```

- **Query**: Decoder'ın şu anki durumu "ne arıyor?"
- **Key**: Encoder çıktıları "ne sunuluyor?"
- **Value**: Encoder çıktıları (key ile genellikle aynı)
- **Score**: Query ile Key arasındaki uyum (dot-product)
- **Weight**: Normalize edilmiş dikkat ağırlıkları


In [ ]:
def attention(query, key, value):
    """
    Luong (dot-product) attention mekanizması.
    
    Boyut gösterimi:
    B  = batch size
    Lq = query uzunluğu (decoder sekans uzunluğu)
    Lk = key/value uzunluğu (encoder sekans uzunluğu)
    dq = query boyutu = dk (key boyutu)
    dv = value boyutu
    
    Args:
        query: Decoder çıktıları  → shape: [B, Lq, dq]
        key:   Encoder çıktıları  → shape: [B, Lk, dk]
        value: Encoder çıktıları  → shape: [B, Lv, dv]  (Lk == Lv)
    
    Returns:
        Weighted sum of values → shape: [B, Lq, dv]
    """
    # Adım 1: Query-Key dot product → alignment scores
    # [B,Lq,dq] @ [B,dk,Lk] = [B, Lq, Lk]  (her query için her key ile uyum skoru)
    scores = query @ key.transpose(1, 2)
    
    # Adım 2: Softmax ile normalize et → attention weights (toplamı 1)
    weights = torch.softmax(scores, dim=-1)  # [B, Lq, Lk]
    
    # Adım 3: Ağırlıklı değer ortalaması al
    # [B, Lq, Lk] @ [B, Lv, dv] = [B, Lq, dv]
    return weights @ value

* `B` = batch size
* `Lq` = max query length in this batch
* `Lk` = max key length in this batch = `Lv` = max value length in this batch
* `dq` = query embedding size = `dk` = key embedding size
* `dv` = value embedding size

**Attention mekanizmalı NMT modeli:**

Temel NMT modelinden farkları:
1. `encoder_outputs` saklanır (sadece hidden state değil)
2. `pad_packed_sequence` ile encoder çıktıları tekrar padding'li forma getirilir
3. Decoder çıktıları attention'dan geçirilir
4. Decoder çıktısı ve attention çıktısı concatenate edilir (2× boyut)
5. Output katmanı `2 × hidden_dim` girdi alır


In [ ]:
class NmtModelWithAttention(nn.Module):
    """
    Luong (dot-product) Attention mekanizmalı Encoder–Decoder NMT modeli.
    
    Temel NmtModel'den fark:
    - Encoder'ın TÜM çıktıları saklanır (sadece final hidden state değil)
    - Her decoder adımı için attention hesaplanır
    - Attention çıktısı decoder çıktısıyla birleştirilir (concatenate)
    """
    def __init__(self, vocab_size, embed_dim=512, pad_id=0, hidden_dim=512,
                 n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.decoder = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        # Attention çıktısı (hidden_dim) + decoder çıktısı (hidden_dim) = 2*hidden_dim
        self.output = nn.Linear(2 * hidden_dim, vocab_size)

    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)
        src_lengths = pair.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(
            src_embeddings, lengths=src_lengths.cpu(),
            batch_first=True, enforce_sorted=False)
        
        # Encoder: hem tüm çıktıları hem final hidden state'i al
        encoder_outputs_packed, hidden_states = self.encoder(src_packed)
        
        # Decoder: encoder'ın hidden state'inden başla (temel model gibi)
        decoder_outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        
        # Packed encoder çıktılarını padding'li tensor'a dönüştür
        encoder_outputs, _ = pad_packed_sequence(encoder_outputs_packed,
                                                 batch_first=True)
        
        # Attention: decoder çıktıları query, encoder çıktıları key ve value
        # Her decoder adımı tüm encoder çıktılarına dikkat edebilir
        attn_output = attention(query=decoder_outputs,
                                key=encoder_outputs, value=encoder_outputs)
        
        # Decoder çıktısı ve attention çıktısını birleştir
        # combined: [B, Lq, 2*hidden_dim]
        combined_output = torch.cat((attn_output, decoder_outputs), dim=-1)
        
        return self.output(combined_output).permute(0, 2, 1)

In [ ]:
torch.manual_seed(42)
nmt_attn_model = NmtModelWithAttention(vocab_size).to(device)

n_epochs = 10
xentropy = nn.CrossEntropyLoss(ignore_index=0)  # <pad> tokenları için loss yok
optimizer = torch.optim.NAdam(nmt_attn_model.parameters(), lr=0.001)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size)
accuracy = accuracy.to(device)

history = train(nmt_attn_model, optimizer, xentropy, accuracy,
                nmt_train_loader, nmt_valid_loader, n_epochs)

In [ ]:
# Attention'lı modeli kaydet
torch.save(nmt_attn_model.state_dict(), "my_nmt_attn_model.pt")

In [ ]:
# Attention'lı model ile orta uzunlukta çeviri
translate(nmt_attn_model, longer_text)

In [ ]:
# Attention'lı model ile daha uzun cümle çeviri
translate(nmt_attn_model, longest_text)

GPU RAM temizleme:

In [ ]:
del_vars(["nmt_attn_model"])

# 8. Ekstra Materyal – Pretrained Embedding Uzayını Keşfetme

### Word Analogies (Kelime Analogileri)

Pretrained embedding'ler güçlü anlamsal ilişkiler öğrenir. Klasik örnek:

**"king" - "man" + "woman" ≈ "queen"**

Bu, embedding uzayında vektör aritmetiği ile gerçekleştirilir:  
`E("queen") - E("king") + E("man") ≈ E("woman")`

Formül: `result = E(token2) - E(token1) + E(token3)`

**Cosine Similarity (Kosinüs Benzerliği):**
- İki vektör arasındaki açının kosinüsü
- -1 (tamamen zıt) ile +1 (tamamen aynı yön) arasında
- Büyüklükten bağımsız yön benzerliği ölçer


In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
# Embedding matrisi: 30522 token × 768 boyut
# .detach() → gradient hesaplamadan ayır (sadece lookup için kullanacağız)
embedding_matrix = model.get_input_embeddings().weight.detach()

In [ ]:
def get_embedding(token):
    """
    Verilen token için BERT embedding vektörünü döndürür.
    
    Args:
        token: Vocabulary'deki token string'i
    
    Returns:
        torch.Tensor: 768 boyutlu embedding vektörü
    """
    token_id = tokenizer.vocab[token]       # Token → ID
    return embedding_matrix[token_id]       # ID → embedding vektörü

# 'hello' token'ının embedding boyutunu kontrol et
get_embedding("hello").shape  # torch.Size([768])def get_embedding(token):
    """
    Verilen token için BERT embedding vektörünü döndürür.
    
    Args:
        token: Vocabulary'deki token string'i
    
    Returns:
        torch.Tensor: 768 boyutlu embedding vektörü
    """
    token_id = tokenizer.vocab[token]       # Token → ID
    return embedding_matrix[token_id]       # ID → embedding vektörü

# 'hello' token'ının embedding boyutunu kontrol et
get_embedding("hello").shape  # torch.Size([768])

Aşağıdaki function, üç token alır, ardından E(token2) - E(token1) + E(token3) hesaplamasını yapar (burada E(token), ilgili token’ın embedding’idir). Daha sonra cosine similarity kullanarak en benzer token embedding’lerini bulur.

İki vektör arasındaki cosine similarity, bu vektörler arasındaki açının kosinüsüdür; dolayısıyla değeri –1 (tamamen zıt yönlü) ile +1 (mükemmel hizalanmış) arasında değişir.

Fonksiyon sonuç olarak (similarity, token) çiftlerinden oluşan bir liste döndürür.

In [ ]:
import torch.nn.functional as F

def find_closest_tokens(token1, token2, token3, top_n=5):
    """
    Analoji sorusu çözer: 'token1 : token2 :: token3 : ?'
    
    Yöntem: E(token2) - E(token1) + E(token3) → hedef vektör
    Bu hedef vektöre en yakın (cosine similarity ile) token'ları bulur.
    
    Args:
        token1, token2, token3: Analoji'nin üç terimi
        top_n: Kaç yakın token döndürülsün
    
    Returns:
        List[Tuple[float, str]]: (benzerlik_skoru, token) çiftleri
    
    Örnek: find_closest_tokens('king', 'queen', 'man')
    Beklenen: [('man', 1.0), ('woman', yüksek_skor), ...]
    """
    E = get_embedding
    # Vektör aritmetiği: hedef vektörü hesapla
    result = E(token2) - E(token1) + E(token3)
    # Hedef vektör ile tüm embedding'ler arasında cosine similarity
    similarities = F.cosine_similarity(result, embedding_matrix)
    # En yüksek benzerlikli top_n token'ı bul
    top_k = torch.topk(similarities, k=top_n)
    return [(sim.item(), tokenizer.decode(idx.item()))
            for sim, idx in zip(top_k.values, top_k.indices)]

Birkaç analoji örneği deneyelim:

In [ ]:
# Klasik word analogy örnekleri
examples = [
    ("king", "queen", "man"),        # king:queen :: man:?  → beklenen: woman
    ("man", "woman", "nephew"),       # man:woman :: nephew:? → beklenen: niece
    ("father", "mother", "son"),      # father:mother :: son:? → beklenen: daughter
    ("man", "woman", "doctor"),       # man:woman :: doctor:? → cinsiyet yanlılığı?
    ("germany", "hitler", "italy"),   # tarihsel bağlam
    ("england", "london", "germany"), # ülke:başkent :: ülke:? → beklenen: berlin
]
for (token1, token2, token3) in examples:
    print(f"{token1} is to {token2} as {token3} is to: ", end="")
    for similarity, token in find_closest_tokens(token1, token2, token3):
        print(f"{token} ({similarity:.1f})", end=" ")
    print()

Gördüğünüz gibi, doğru cevap genellikle en yakın sonuçlar arasında yer alır.

Bununla birlikte, başka örneklerle denemeler yaparsanız bunun yalnızca nispeten basit örneklerde iyi çalıştığını fark edersiniz. Embeddings her zaman bu kadar kolay yorumlanabilir değildir.

# 9. Egzersiz Çözümleri

Bu bölümde bölümün kilit kavramları üzerine soru-cevap formatında özet sunulmaktadır.


## 9.1 Temel Sorular ve Cevaplar

**1. Stateless vs Stateful RNN ne zaman tercih edilir?**

**Stateless RNN:** Sadece `window_length` kadar bağlamı öğrenebilir. Uygulama açısından basit ve IID (bağımsız ve özdeş dağılımlı) batchler kullandığı için gradient descent ile iyi çalışır.

**Stateful RNN:** `window_length`'ten uzun kalıpları öğrenebilir. Ancak:
- Dataset hazırlamak çok daha zordur (batchler ardışık olmalı)
- Batchler IID değildir → gradient descent'in performance garantisi azalır
- Her zaman daha iyi sonuç vermez

---

**2. Encoder–Decoder vs düz Sequence-to-Sequence RNN (çeviri için)?**

"Je vous en prie" → Kelime kelime: "I you in pray" (anlamsız!)  
Çeviri cümle yapısına bağımlı olduğundan:

- **Düz RNN:** İlk kelimeyi okur okumaz çevirmeye başlar → bağlam eksik
- **Encoder–Decoder:** Tüm cümleyi okur, sonra çevirir → çok daha doğru

---

**3. Değişken uzunluklu sekanslar nasıl ele alınır?**

- **Padding:** Kısa sekansları uzun olana kadar sıfır ile doldur
- **Masking:** Model padding token'larını yoksayar (attention mask)
- **pack_padded_sequence:** Padding'i RNN'e vermeden es geçer
- **drop_last=True:** Batchler farklı boyutta olmasın diye eksik son batchi at

---

**4. Beam Search neden greedy decoding'den iyidir?**

Greedy: Her adımda en iyi tek seçim → lokal optimum tuzağı  
Beam Search: k en iyi yolu paralel takip eder → daha iyi global çözüm  
Büyük k → daha iyi kalite, daha yüksek hesaplama maliyeti

---

**5. Attention mekanizması ne sağlar?**

- Encoder'ın tüm çıktılarına erişim → bilgi darboğazını ortadan kaldırır
- Decoder her adımda kaynak sekansın ilgili kısmına odaklanır
- Attention ağırlıkları yorumlanabilir hizalama (alignment) haritaları üretir
- Transformer'ın multi-head attention'ının temel yapı taşı

---

**6. Sampled Softmax nedir?**

Çok büyük vocabulary'lerde (örn. 100.000 kelime) tam softmax hesaplamak yavaştır.  
**Sampled Softmax:** Sadece doğru kelime + rastgele seçilmiş yanlış kelimelerin logit'lerini kullanarak loss'u yaklaşık hesaplar → training hızlanır.  
Inference sırasında normal softmax kullanılır (tüm vocabulary üzerinde).
